# Data Augmentation - Explained

## What This Notebook Covers

This notebook explores **data augmentation** - techniques to artificially expand your training dataset by applying transformations to existing images. Augmentation is one of the most effective ways to improve model generalization.

### Why Data Augmentation?

| Problem | Without Augmentation | With Augmentation |
|---------|---------------------|------------------|
| **Overfitting** | Model memorizes training data | Model learns general patterns |
| **Limited data** | Need more images | Artificially expand dataset |
| **Position invariance** | Model fails on shifted images | Model handles variations |

### Example: Training vs Validation Gap

Without augmentation after 20 epochs:
```
Training accuracy:   99.9%
Validation accuracy: 92.4%  ← Big gap = overfitting!
```

### Augmentation Techniques We'll Cover

1. **Going Wider** - Wider ResNet models
2. **Global Average Pooling** - Reduce parameters
3. **Basic Augmentation** - RandomCrop, RandomHorizontalFlip
4. **Test Time Augmentation (TTA)** - Augment at inference
5. **Random Erase** - Randomly erase parts of image
6. **Random Copy** - Copy patches within image
7. **Dropout** - Randomly zero activations

Let's dive in!

---
## Part 1: Setup and Imports

In [ ]:
#|default_exp augment
# This cell marks that code with #|export will be exported to miniai/augment.py

In [ ]:
#|export
import torch, random
import fastcore.all as fc

from torch import nn
from torch.nn import init

# Our custom modules from previous notebooks
from miniai.datasets import *
from miniai.conv import *
from miniai.learner import *
from miniai.activations import *
from miniai.init import *
from miniai.sgd import *
from miniai.resnet import *

In [ ]:
# Standard imports
import pickle, gzip, math, os, time, shutil
import matplotlib as mpl, numpy as np, matplotlib.pyplot as plt
from collections.abc import Mapping
from pathlib import Path
from operator import attrgetter, itemgetter
from functools import partial
from copy import copy
from contextlib import contextmanager

# PyTorch imports
import torchvision.transforms.functional as TF
import torch.nn.functional as F
from torch import tensor, optim
from torch.utils.data import DataLoader, default_collate
from torch.optim import lr_scheduler
from torcheval.metrics import MulticlassAccuracy
from datasets import load_dataset, load_dataset_builder

from fastcore.test import test_close
from torch import distributions  # For dropout implementation

# Configure display
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray'

In [ ]:
# ============================================================================
# INTERACTIVE VISUALIZATIONS -- setup (run this cell ONCE).
# Each "🎮 Interactive" cell below loads a standalone HTML file from the
# published copy on GitHub Pages, in an isolated <iframe> (its CSS/JS can't leak
# into the notebook). It is shown FULL WIDTH and auto-fits its content height.
# No local files needed -- the visualizations are read from GitHub.
# ============================================================================
from IPython.display import HTML

def show_viz(path, height="600px"):
    """Embed an interactive visualization full width; it auto-fits its height.
    `path` may be a bare 'interactive_viz/<file>.html' (resolved to the GitHub
    Pages copy) or a full https URL."""
    base = "https://shammun.github.io/shammunul-fastai-notes/notebooks/"
    if not path.startswith("http"):
        path = base + path
    return HTML(
        f'<iframe src="{path}" loading="lazy" allowfullscreen '
        f'style="width:100%;height:{height};border:1px solid #dde5f2;border-radius:12px;'
        f'box-shadow:0 8px 24px rgba(123,92,214,.12);background:#fff;"></iframe>'
        '<script>addEventListener("message",function(e){'
        'if(e.data&&e.data.type==="ce-frame-height"&&e.data.height>50){'
        'var fs=document.querySelectorAll("iframe");for(var i=0;i<fs.length;i++){'
        'if(fs[i].contentWindow===e.source){fs[i].style.height=e.data.height+"px";break;}}}});</script>'
    )

---
## Part 2: Data Setup

In [ ]:
# Dataset configuration
xl, yl = 'image', 'label'
name = "fashion_mnist"
bs = 1024
xmean, xstd = 0.28, 0.35  # Normalization constants

# Transform: normalize images
@inplace
def transformi(b): 
    b[xl] = [(TF.to_tensor(o) - xmean) / xstd for o in b[xl]]

# Load dataset
dsd = load_dataset(name)
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=fc.defaults.cpus)

In [ ]:
# Setup callbacks and model configuration
metrics = MetricsCB(accuracy=MulticlassAccuracy())
astats = ActivationStats(fc.risinstance(GeneralRelu))
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), astats]

# GeneralRelu activation
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)

# Kaiming initialization
iw = partial(init_weights, leaky=0.1)

In [ ]:
# Training configuration
set_seed(42)
lr, epochs = 6e-2, 5

---
## Part 3: Going Wider

Before we explore augmentation, let's try making our model wider (more channels per layer).

In [ ]:
def get_model(act=nn.ReLU, nfs=(16, 32, 64, 128, 256, 512), norm=nn.BatchNorm2d):
    """
    Create a wider ResNet model.
    
    Note: This version uses more channels (16→512 vs 8→256 before).
    The first layer uses a 5x5 kernel for better feature capture.
    
    Args:
        act: Activation function class
        nfs: Tuple of filter counts (default: wider than before)
        norm: Normalization layer class
    
    Returns:
        nn.Sequential model
    """
    # First ResBlock with 5x5 kernel (larger receptive field)
    layers = [ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm)]
    
    # Add ResBlocks with stride=2 (halving spatial dimensions)
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) 
               for i in range(len(nfs)-1)]
    
    # Classification head
    layers += [nn.Flatten(), nn.Linear(nfs[-1], 10, bias=False), nn.BatchNorm1d(10)]
    
    return nn.Sequential(*layers)

In [ ]:
# Train wider model
lr = 1e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]

model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)

In [ ]:
learn.fit(epochs)

---
## Part 4: Global Average Pooling

Instead of flattening the final feature map directly, we can use **Global Average Pooling (GAP)** to reduce each channel to a single value.

In [ ]:
class GlobalAvgPool(nn.Module):
    """
    Global Average Pooling.
    
    Takes the mean across spatial dimensions (height and width),
    reducing each channel to a single value.
    
    Input:  (batch, channels, height, width)
    Output: (batch, channels)
    
    Benefits:
    - Reduces parameters (no flattening of spatial dims)
    - Works with any input size
    - Acts as structural regularization
    """
    def forward(self, x): 
        # Take mean over last two dimensions (height, width)
        # (-2, -1) means second-to-last and last dimensions
        return x.mean((-2, -1))

### Why Global Average Pooling?

```
Without GAP:                        With GAP:

Feature map: [batch, 512, 2, 2]     Feature map: [batch, 512, 2, 2]
      │                                    │
  Flatten                              GlobalAvgPool
      │                                    │
[batch, 512*2*2 = 2048]             [batch, 512]
      │                                    │
Linear(2048, 10)                    Linear(512, 10)
= 20,480 params                     = 5,120 params (4x fewer!)
```

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. It loads ./interactive_viz/gap_3d_explorer.html
# at full width, sized to fit the visualization (setup cell near the top required).
# 3D: watch each channel's H×W plane collapse to ONE number (drag to rotate,
# click a plane to average it) — and see why GAP needs 4× fewer classifier params.
# ============================================================================
show_viz("interactive_viz/gap_3d_explorer.html", height="660px")

In [ ]:
def get_model2(act=nn.ReLU, nfs=(16, 32, 64, 128, 256), norm=nn.BatchNorm2d):
    """
    Model with an extra ResBlock before GlobalAvgPool.
    
    Architecture:
    - ResBlocks with stride=2 until 256 channels at 2x2
    - One more ResBlock (256→512) WITHOUT stride (stays 2x2)
    - GlobalAvgPool: 2x2 → 1x1 (single value per channel)
    """
    layers = [ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) 
               for i in range(len(nfs)-1)]
    # Extra ResBlock without stride, then GlobalAvgPool
    layers += [ResBlock(256, 512, act=act, norm=norm), GlobalAvgPool()]
    layers += [nn.Linear(512, 10, bias=False), nn.BatchNorm1d(10)]
    return nn.Sequential(*layers)

In [ ]:
# =============================================================================
# get_model2: A ResNet Model with GlobalAvgPool — Fully Explained
# =============================================================================
#
# WHAT IS THIS MODEL?
# -------------------
# This is a Residual Network (ResNet) designed for image classification.
# Specifically, it classifies 28x28 grayscale images (like Fashion MNIST)
# into 10 categories (e.g., T-shirt, Trouser, Pullover, Dress, etc.).
#
# WHY RESNET?
# -----------
# In a plain deep network, stacking more layers often makes performance
# WORSE (not better). This is called the "degradation problem." ResNets
# solve this with "skip connections" — shortcuts that let the input bypass
# convolutional layers and get added directly to the output. This makes it
# easy for extra layers to learn "do nothing" (identity) if they can't
# improve things, so adding more layers can never hurt.
#
# HOW IS get_model2 DIFFERENT FROM get_model?
# --------------------------------------------
# get_model:  Downsamples all the way to 1x1 using stride=2 convolutions,
#             then uses Flatten() + Linear to classify.
#
# get_model2: Stops downsampling at 2x2, adds an EXTRA ResBlock (256→512)
#             that keeps the 2x2 size, then uses GlobalAvgPool to go from
#             2x2 → 1x1. This is the more modern approach used in real
#             ResNets because GlobalAvgPool works on ANY spatial size,
#             making the model flexible to different input resolutions.
#
# =============================================================================
#
# SIZE TRACKING THROUGH THE ENTIRE MODEL
# =======================================
#
# We track the tensor shape as: (batch, channels, height, width)
# Using batch=B for brevity:
#
#   INPUT IMAGE:                         (B,   1, 28, 28)
#      ↓
#   ResBlock(1→16, ks=5, stride=1)   →  (B,  16, 28, 28)   [same spatial size]
#      ↓
#   ResBlock(16→32, stride=2)        →  (B,  32, 14, 14)   [halved: 28→14]
#      ↓
#   ResBlock(32→64, stride=2)        →  (B,  64,  7,  7)   [halved: 14→7]
#      ↓
#   ResBlock(64→128, stride=2)       →  (B, 128,  4,  4)   [halved: 7→4, ceil]
#      ↓
#   ResBlock(128→256, stride=2)      →  (B, 256,  2,  2)   [halved: 4→2]
#      ↓
#   ResBlock(256→512, stride=1)      →  (B, 512,  2,  2)   [same spatial size]
#      ↓
#   GlobalAvgPool()                  →  (B, 512)            [2x2 averaged to 1 number per channel, then squeezed]
#      ↓
#   Linear(512→10)                   →  (B,  10)            [512 features mapped to 10 class scores]
#      ↓
#   BatchNorm1d(10)                  →  (B,  10)            [normalize the 10 scores]
#
#   OUTPUT: 10 scores (one per class). Highest score = predicted class.
#
# =============================================================================


def get_model2(act=nn.ReLU, nfs=(16, 32, 64, 128, 256), norm=nn.BatchNorm2d):
    # -------------------------------------------------------------------------
    # FUNCTION ARGUMENTS — What each one controls:
    # -------------------------------------------------------------------------
    #
    # act = nn.ReLU
    #   The activation function used after each convolutional layer.
    #   Activation functions introduce "non-linearity" — without them,
    #   stacking layers would be no better than a single layer (because
    #   a chain of linear transformations is just another linear transformation).
    #   nn.ReLU (Rectified Linear Unit) is the most common choice:
    #       ReLU(x) = max(0, x)   →   keeps positives, zeros out negatives.
    #   Default is nn.ReLU, but you can pass in alternatives like act_gr
    #   (GeneralRelu with leak=0.1 and sub=0.4) for potentially better training.
    #
    # nfs = (16, 32, 64, 128, 256)
    #   "nfs" stands for "number of filters" (i.e., number of output channels)
    #   at each stage of the network. Each number defines how many feature
    #   maps (channels) a ResBlock outputs. The pattern of doubling
    #   (16→32→64→128→256) is a classic design choice in CNNs:
    #       - Early layers: fewer channels (cheap, capture simple patterns
    #         like edges and textures)
    #       - Later layers: more channels (expensive, capture complex patterns
    #         like shapes and object parts)
    #   As spatial size shrinks (28→14→7→4→2), we compensate by increasing
    #   channels, so the total information capacity stays roughly balanced.
    #
    # norm = nn.BatchNorm2d
    #   The normalization layer applied after each convolution.
    #   BatchNorm2d normalizes the outputs of each channel to have roughly
    #   mean=0 and std=1 across the batch. This helps because:
    #       1. Stabilizes training (prevents activations from exploding/vanishing)
    #       2. Allows higher learning rates (faster training)
    #       3. Acts as mild regularization (reduces overfitting slightly)
    #   The "2d" means it works on 2D spatial data (images). It has 2 learnable
    #   parameters per channel: gamma (scale) and beta (shift).
    #
    # -------------------------------------------------------------------------

    # =========================================================================
    # LAYER 1: First ResBlock — The "Entry" Layer
    # =========================================================================
    #
    # ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm)
    #
    # What each argument means:
    #   1       → ni (number of input channels) = 1
    #             Our input is a grayscale image, which has only 1 color channel.
    #             (RGB images would have 3 channels.)
    #
    #   16      → nf (number of output filters/channels) = 16
    #             This layer creates 16 different "feature maps." Each feature map
    #             is like a filtered version of the image that detects a specific
    #             pattern (e.g., horizontal edges, vertical edges, corners, etc.).
    #
    #   ks=5    → kernel_size = 5 (a 5×5 convolutional filter)
    #             The first layer uses a LARGER kernel than the default (3×3).
    #             Why? The first layer sees raw pixels, and a 5×5 window captures
    #             more context in a single look — it can detect slightly larger
    #             patterns right away. Later layers use 3×3 (the default) because
    #             they already see higher-level features from previous layers.
    #
    #   stride=1 → The convolution slides 1 pixel at a time.
    #              This means the output has the SAME spatial size as the input.
    #              We don't want to lose spatial resolution at the very first layer
    #              because we want to preserve fine details from the original image
    #              for the subsequent layers to work with.
    #
    #   act=act  → Uses the activation function passed to get_model2 (default: ReLU)
    #
    #   norm=norm → Uses the normalization layer passed to get_model2 (default: BatchNorm2d)
    #
    # Inside this ResBlock, here's what happens step by step:
    #
    #   MAIN PATH (learns the residual F(x)):
    #     Conv2d(1→16, ks=5, stride=1) + BatchNorm2d(16) + ReLU
    #         (B, 1, 28, 28) → (B, 16, 28, 28)
    #     Conv2d(16→16, ks=5, stride=1) + BatchNorm2d(16)  [NO activation yet]
    #         (B, 16, 28, 28) → (B, 16, 28, 28)
    #
    #   SHORTCUT PATH (transforms x to match dimensions):
    #     Since ni=1 ≠ nf=16, a 1×1 Conv is needed: Conv2d(1→16, ks=1)
    #         (B, 1, 28, 28) → (B, 16, 28, 28)
    #     Since stride=1, no pooling is needed (spatial size stays the same).
    #
    #   COMBINE:
    #     output = ReLU( main_path_output + shortcut_output )
    #         (B, 16, 28, 28) + (B, 16, 28, 28) → (B, 16, 28, 28)
    #
    # Size change: (B, 1, 28, 28) → (B, 16, 28, 28)
    #   Channels: 1 → 16  |  Spatial: 28×28 → 28×28 (unchanged)
    #
    layers = [ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm)]

    # =========================================================================
    # LAYERS 2-5: Downsampling ResBlocks — The "Body" of the Network
    # =========================================================================
    #
    # This loop creates 4 ResBlocks that progressively:
    #   - DOUBLE the number of channels (more features, more capacity)
    #   - HALVE the spatial dimensions (compress spatial info, keep what matters)
    #
    # The loop iterates over consecutive pairs in nfs = (16, 32, 64, 128, 256):
    #   i=0: ResBlock(nfs[0]→nfs[1]) = ResBlock(16→32,   stride=2)
    #   i=1: ResBlock(nfs[1]→nfs[2]) = ResBlock(32→64,   stride=2)
    #   i=2: ResBlock(nfs[2]→nfs[3]) = ResBlock(64→128,  stride=2)
    #   i=3: ResBlock(nfs[3]→nfs[4]) = ResBlock(128→256, stride=2)
    #
    # range(len(nfs)-1) = range(5-1) = range(4) = [0, 1, 2, 3]
    # So we get exactly 4 ResBlocks, one for each consecutive pair.
    #
    # stride=2 means the second convolution inside each ResBlock moves its
    # filter 2 pixels at a time instead of 1, which halves the spatial size.
    # (Think of it like reading every other word — you cover the same text
    # in half the space.)
    #
    # Note: ks is NOT specified here, so it defaults to ks=3 (3×3 kernels).
    # After the first layer has already extracted basic features, 3×3 is
    # sufficient and computationally cheaper than 5×5.
    #
    # --- Detailed size tracking for each ResBlock in this loop ---
    #
    # ResBlock(16→32, stride=2):
    #   Main path:
    #     Conv2d(16→32, ks=3, stride=1): (B, 16, 28, 28) → (B, 32, 28, 28)
    #     Conv2d(32→32, ks=3, stride=2): (B, 32, 28, 28) → (B, 32, 14, 14)
    #   Shortcut path:
    #     AvgPool2d(2):      (B, 16, 28, 28) → (B, 16, 14, 14)  [halve spatial]
    #     Conv2d(16→32, 1×1): (B, 16, 14, 14) → (B, 32, 14, 14) [match channels]
    #   Combine: (B, 32, 14, 14) + (B, 32, 14, 14) → (B, 32, 14, 14)
    #   Size change: (B, 16, 28, 28) → (B, 32, 14, 14)
    #
    # ResBlock(32→64, stride=2):
    #   Main path:
    #     Conv2d(32→64, ks=3, stride=1): (B, 32, 14, 14) → (B, 64, 14, 14)
    #     Conv2d(64→64, ks=3, stride=2): (B, 64, 14, 14) → (B, 64,  7,  7)
    #   Shortcut path:
    #     AvgPool2d(2):      (B, 32, 14, 14) → (B, 32, 7, 7)
    #     Conv2d(32→64, 1×1): (B, 32,  7,  7) → (B, 64, 7, 7)
    #   Combine: (B, 64, 7, 7) + (B, 64, 7, 7) → (B, 64, 7, 7)
    #   Size change: (B, 32, 14, 14) → (B, 64, 7, 7)
    #
    # ResBlock(64→128, stride=2):
    #   Main path:
    #     Conv2d(64→128, ks=3, stride=1): (B, 64, 7, 7) → (B, 128, 7, 7)
    #     Conv2d(128→128, ks=3, stride=2): (B, 128, 7, 7) → (B, 128, 4, 4)
    #       (7/2 = 3.5, but with padding=1 and ceil_mode, output = 4)
    #   Shortcut path:
    #     AvgPool2d(2, ceil_mode=True): (B, 64, 7, 7) → (B, 64, 4, 4)
    #     Conv2d(64→128, 1×1):          (B, 64, 4, 4) → (B, 128, 4, 4)
    #   Combine: (B, 128, 4, 4) + (B, 128, 4, 4) → (B, 128, 4, 4)
    #   Size change: (B, 64, 7, 7) → (B, 128, 4, 4)
    #
    # ResBlock(128→256, stride=2):
    #   Main path:
    #     Conv2d(128→256, ks=3, stride=1): (B, 128, 4, 4) → (B, 256, 4, 4)
    #     Conv2d(256→256, ks=3, stride=2): (B, 256, 4, 4) → (B, 256, 2, 2)
    #   Shortcut path:
    #     AvgPool2d(2):       (B, 128, 4, 4) → (B, 128, 2, 2)
    #     Conv2d(128→256, 1×1): (B, 128, 2, 2) → (B, 256, 2, 2)
    #   Combine: (B, 256, 2, 2) + (B, 256, 2, 2) → (B, 256, 2, 2)
    #   Size change: (B, 128, 4, 4) → (B, 256, 2, 2)
    #
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
               for i in range(len(nfs)-1)]

    # =========================================================================
    # LAYER 6: Extra ResBlock WITHOUT Downsampling — More Processing Power
    # =========================================================================
    #
    # ResBlock(256, 512, act=act, norm=norm)
    #
    # Notice: stride is NOT specified, so it defaults to stride=1.
    # This means spatial dimensions stay the SAME (2×2 → 2×2).
    #
    # Why add this extra block?
    #   At this point we're at 2×2 spatial size — very small. If we used
    #   stride=2 again, we'd go to 1×1 and lose all spatial relationships.
    #   Instead, we keep the 2×2 size and just increase the channels from
    #   256 to 512. This gives the network more "thinking capacity" to
    #   combine and refine the features it has learned so far, right before
    #   making its final classification decision.
    #
    # Inside this ResBlock:
    #   Main path:
    #     Conv2d(256→512, ks=3, stride=1): (B, 256, 2, 2) → (B, 512, 2, 2)
    #     Conv2d(512→512, ks=3, stride=1): (B, 512, 2, 2) → (B, 512, 2, 2)
    #   Shortcut path:
    #     No pooling needed (stride=1, spatial size unchanged).
    #     Conv2d(256→512, 1×1): (B, 256, 2, 2) → (B, 512, 2, 2)
    #       (needed because channels change from 256 to 512)
    #   Combine: (B, 512, 2, 2) + (B, 512, 2, 2) → (B, 512, 2, 2)
    #
    # Size change: (B, 256, 2, 2) → (B, 512, 2, 2)
    #   Channels: 256 → 512  |  Spatial: 2×2 → 2×2 (unchanged)
    #
    # =========================================================================
    # GlobalAvgPool() — Collapse Spatial Dimensions
    # =========================================================================
    #
    # GlobalAvgPool takes EACH of the 512 channels (each a 2×2 grid of 4 values)
    # and computes their AVERAGE, reducing each channel to a single number.
    #
    # Conceptually for one channel:
    #   [[0.5, 0.3],
    #    [0.8, 0.4]]  →  mean(0.5, 0.3, 0.8, 0.4) = 0.5
    #
    # This happens independently for all 512 channels:
    #   (B, 512, 2, 2) → (B, 512)
    #
    # Implementation (typically):
    #   class GlobalAvgPool(nn.Module):
    #       def forward(self, x):
    #           return x.mean((-2, -1))  # average over the last 2 dims (H and W)
    #
    # Why use GlobalAvgPool instead of just more stride=2 convolutions?
    #   1. FLEXIBILITY: Works on ANY spatial size (2×2, 4×4, 7×7, etc.),
    #      so the same model can handle different input image sizes.
    #   2. NO PARAMETERS: Averaging has zero learnable parameters,
    #      which means less risk of overfitting.
    #   3. ELEGANCE: Cleanly separates "feature extraction" (the ResBlocks)
    #      from "spatial reduction" (GlobalAvgPool), making the architecture
    #      easier to reason about and modify.
    #
    # Size change: (B, 512, 2, 2) → (B, 512)
    #   The 2×2 spatial grid is gone; we now have a flat vector of 512 features.
    #
    layers += [ResBlock(256, 512, act=act, norm=norm), GlobalAvgPool()]

    # =========================================================================
    # CLASSIFICATION HEAD: Linear + BatchNorm1d
    # =========================================================================
    #
    # nn.Linear(512, 10, bias=False)
    #   A fully connected (linear) layer that maps 512 features → 10 class scores.
    #
    #   What it does mathematically:
    #       output = input @ weight.T     (matrix multiplication, no bias)
    #       where weight has shape (10, 512)
    #
    #   Why 512 → 10?
    #       512 = the number of channels from the last ResBlock
    #       10  = the number of classes in Fashion MNIST
    #       Each of the 10 outputs is a "score" for one class. The class
    #       with the highest score is the model's prediction.
    #
    #   Why bias=False?
    #       Because the next layer (BatchNorm1d) already has its own bias
    #       parameter (called "beta"). Having bias in BOTH the Linear layer
    #       and BatchNorm would be redundant — the two biases would just
    #       combine into one effective value, wasting parameters.
    #
    #   Parameters: 512 × 10 = 5,120 weights (no bias)
    #
    #   Size change: (B, 512) → (B, 10)
    #
    # nn.BatchNorm1d(10)
    #   Normalizes the 10 output scores across the batch.
    #
    #   The "1d" is because our data is now 1-dimensional per sample (just
    #   10 numbers), not 2D spatial data like images. Compare:
    #       BatchNorm2d: for (B, C, H, W) tensors (images with spatial dims)
    #       BatchNorm1d: for (B, C) tensors (flat vectors with no spatial dims)
    #
    #   Why normalize the final outputs (logits)?
    #       It helps stabilize training by keeping the logits (raw class scores)
    #       in a well-behaved range. Without it, logits can drift to very
    #       large or very small values, making the loss function behave poorly.
    #
    #   Parameters: 2 × 10 = 20 (10 gammas + 10 betas)
    #
    #   Size change: (B, 10) → (B, 10)  [shape unchanged, values normalized]
    #
    layers += [nn.Linear(512, 10, bias=False), nn.BatchNorm1d(10)]

    # =========================================================================
    # ASSEMBLE AND RETURN
    # =========================================================================
    #
    # nn.Sequential(*layers)
    #   Wraps all the layers into a single module that runs them in order.
    #   The * operator "unpacks" the list, so:
    #       nn.Sequential(*[layer1, layer2, layer3])
    #   is the same as:
    #       nn.Sequential(layer1, layer2, layer3)
    #
    #   When you call model(x), PyTorch passes x through each layer in
    #   sequence: output = layer9(layer8(layer7(...(layer1(x))...)))
    #
    #   Note: Unlike get_model which calls .to(def_device), this version
    #   does NOT move the model to a specific device (CPU/GPU) here.
    #   The caller is responsible for moving it to the right device.
    #
    return nn.Sequential(*layers)


# =============================================================================
# COMPLETE ARCHITECTURE SUMMARY TABLE
# =============================================================================
#
# ┌────┬──────────────────────────────────┬────────────────────┬───────────────────┐
# │ #  │ Layer                            │ Input Shape        │ Output Shape      │
# ├────┼──────────────────────────────────┼────────────────────┼───────────────────┤
# │  1 │ ResBlock(1→16, ks=5, stride=1)   │ (B,   1, 28, 28)  │ (B,  16, 28, 28)  │
# │  2 │ ResBlock(16→32, stride=2)        │ (B,  16, 28, 28)  │ (B,  32, 14, 14)  │
# │  3 │ ResBlock(32→64, stride=2)        │ (B,  32, 14, 14)  │ (B,  64,  7,  7)  │
# │  4 │ ResBlock(64→128, stride=2)       │ (B,  64,  7,  7)  │ (B, 128,  4,  4)  │
# │  5 │ ResBlock(128→256, stride=2)      │ (B, 128,  4,  4)  │ (B, 256,  2,  2)  │
# │  6 │ ResBlock(256→512, stride=1)      │ (B, 256,  2,  2)  │ (B, 512,  2,  2)  │
# │  7 │ GlobalAvgPool()                  │ (B, 512,  2,  2)  │ (B, 512)          │
# │  8 │ Linear(512→10, bias=False)       │ (B, 512)          │ (B,  10)          │
# │  9 │ BatchNorm1d(10)                  │ (B,  10)          │ (B,  10)          │
# └────┴──────────────────────────────────┴────────────────────┴───────────────────┘
#
# WHAT EACH STAGE IS DOING (Big Picture):
#
#   Layers 1:     "LOOK" — Extract basic features from raw pixels (edges, textures)
#   Layers 2-5:   "THINK" — Build increasingly abstract features while compressing
#                  spatial info (shapes → parts → objects)
#   Layer 6:      "REFINE" — Extra processing with 512 channels at 2×2
#   Layer 7:      "SUMMARIZE" — Collapse spatial dimensions to get one number per feature
#   Layers 8-9:   "DECIDE" — Map features to class scores and normalize
#
# =============================================================================

### Model Summary with FLOPS

Let's update our summary function to also show computational cost (FLOPS).

In [ ]:
#|export
def _flops(x, h, w):
    """
    Estimate FLOPS for a parameter tensor.
    
    FLOPS (Floating Point Operations) measures computational cost.
    For convolutions, each output pixel requires kernel_size^2 * in_channels
    multiply-adds, done for each output pixel and output channel.
    
    Args:
        x: Parameter tensor
        h, w: Output height and width
    
    Returns:
        Estimated number of operations
    """
    if x.dim() < 3: 
        return x.numel()  # Linear layer: just count parameters
    if x.dim() == 4: 
        return x.numel() * h * w  # Conv: params * output spatial size

@fc.patch
def summary(self: Learner):
    """
    Print model summary with parameter counts and FLOPS.
    """
    res = '|Module|Input|Output|Num params|MFLOPS|\n|--|--|--|--|--|\n'
    totp, totf = 0, 0  # Total params and flops
    
    def _f(hook, mod, inp, outp):
        nonlocal res, totp, totf
        # Count parameters
        nparms = sum(o.numel() for o in mod.parameters())
        totp += nparms
        # Estimate FLOPS
        *_, h, w = outp.shape
        flops = sum(_flops(o, h, w) for o in mod.parameters()) / 1e6  # In millions
        totf += flops
        res += f'|{type(mod).__name__}|{tuple(inp[0].shape)}|{tuple(outp.shape)}|{nparms}|{flops:.1f}|\n'
    
    with Hooks(self.model, _f) as hooks: 
        self.fit(1, lr=1, cbs=SingleBatchCB())
    
    print(f"Total params: {totp:,}, Total MFLOPS: {totf:.1f}")
    if fc.IN_NOTEBOOK:
        from IPython.display import Markdown
        return Markdown(res)
    else: 
        print(res)

In [ ]:
# Show model summary with FLOPS
TrainLearner(get_model2(), dls, F.cross_entropy, lr=lr, cbs=[DeviceCB()]).summary()

In [ ]:
# Train model with GlobalAvgPool
set_seed(42)
model = get_model2(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(epochs)

### Simpler Version: GAP Directly After ResBlocks

In [ ]:
def get_model3(act=nn.ReLU, nfs=(16, 32, 64, 128, 256), norm=nn.BatchNorm2d):
    """
    Simpler model: GlobalAvgPool right after ResBlocks.
    
    No extra 256→512 block, just pool the 256-channel features.
    """
    layers = [ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) 
               for i in range(len(nfs)-1)]
    # Directly apply GAP and classify
    layers += [GlobalAvgPool(), nn.Linear(256, 10, bias=False), nn.BatchNorm1d(10)]
    return nn.Sequential(*layers)

In [ ]:
TrainLearner(get_model3(), dls, F.cross_entropy, lr=lr, cbs=[DeviceCB()]).summary()

In [ ]:
set_seed(42)
model = get_model3(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(epochs)

### Even Simpler: First Layer as Plain Conv

In [ ]:
def get_model4(act=nn.ReLU, nfs=(16, 32, 64, 128, 256), norm=nn.BatchNorm2d):
    """
    Simplest version: first layer is plain conv, not ResBlock.
    
    Since the first layer has 1 input channel (grayscale),
    there's no benefit to a skip connection (channels must change).
    """
    # First layer: plain conv (1→16 channels)
    layers = [conv(1, 16, ks=5, stride=1, act=act, norm=norm)]
    # ResBlocks for the rest
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) 
               for i in range(len(nfs)-1)]
    layers += [GlobalAvgPool(), nn.Linear(256, 10, bias=False), nn.BatchNorm1d(10)]
    return nn.Sequential(*layers)

In [ ]:
# Compare parameter shapes
print("Model3 first layer params:", [o.shape for o in get_model3()[0].parameters()])
print("Model4 first layer params:", [o.shape for o in get_model4()[0].parameters()])

In [ ]:
TrainLearner(get_model4(), dls, F.cross_entropy, lr=lr, cbs=[DeviceCB()]).summary()

In [ ]:
set_seed(42)
model = get_model4(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(epochs)

---
## Part 5: Data Augmentation Basics

Now let's add data augmentation to improve generalization.

### The Overfitting Problem

After 20 epochs without augmentation:
```
Training accuracy:   99.9%  ← Memorized training data
Validation accuracy: 92.4%  ← Doesn't generalize well
```

With BatchNorm, weight decay doesn't really help with regularization. We need data augmentation!

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. It loads ./interactive_viz/why_augment_overfitting.html
# at full width, sized to fit the visualization (setup cell near the top required).
# WHY augment: re-run 20 epochs at different augmentation strengths and watch
# the train/valid overfitting gap open (0%) and close (~50-80%).
# ============================================================================
show_viz("interactive_viz/why_augment_overfitting.html", height="560px")

In [ ]:
from torchvision import transforms

In [ ]:
def tfm_batch(b, tfm_x=fc.noop, tfm_y=fc.noop):
    """
    Apply transformations to a batch.
    
    Args:
        b: Batch tuple (images, labels)
        tfm_x: Transform to apply to images (default: identity)
        tfm_y: Transform to apply to labels (default: identity)
    
    Returns:
        Transformed (images, labels) tuple
    """
    return tfm_x(b[0]), tfm_y(b[1])

In [ ]:
# Define augmentation transforms
tfms = nn.Sequential(
    # RandomCrop: Randomly crop a 28x28 region from a 36x36 padded image
    # padding=4 means add 4 pixels on each side → 28+4+4 = 36
    # Then crop back to 28x28 from a random position
    transforms.RandomCrop(28, padding=4),
    
    # RandomHorizontalFlip: 50% chance to flip horizontally
    transforms.RandomHorizontalFlip()
)

# Create callback that applies augmentation only during training
# on_val=False means don't augment validation data
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)

# Test the augmentation
model = get_model()
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=[SingleBatchCB(), augcb])

In [ ]:
# Run one batch to see augmentation
learn.fit(1)

# Test Time Augmentation (TTA) — Explained from Scratch

## What Problem Does TTA Solve?

Imagine you trained a model to classify images of clothing (Fashion MNIST). After training is done, you run the model on the **validation set** — each image goes through the model **once**, and you get a prediction. Simple enough.

But here's the thing: if you **flip** that same image horizontally (like a mirror), the model might give a **slightly different prediction**. A T-shirt facing left is still a T-shirt when it's facing right — but the model might be a little more confident about one version than the other, purely because of the specific patterns it happened to learn during training.

**What if we could use BOTH versions (original + flipped) to make a better prediction?**

That's exactly what Test Time Augmentation does.

---

## The Core Idea (No Code Yet)

TTA is surprisingly simple. Here's the recipe:

1. **Take the original test image** → feed it through the model → get prediction scores
2. **Take a transformed version** (e.g., horizontally flipped) → feed it through the **same model** → get prediction scores
3. **Average** the two sets of prediction scores together
4. Use the averaged scores to make the **final prediction**

That's it. By averaging multiple "opinions" from the same model (each looking at a different version of the image), we smooth out individual mistakes. It's like asking two doctors instead of one — the **consensus** is usually more reliable.

**The best part?** We don't retrain anything. TTA is a **free accuracy boost** at inference time. The only cost is that inference takes a bit longer (we run the model multiple times per image instead of once).

### A Simple Analogy

Imagine you're trying to read a blurry sign from a distance. If you look at it from **one angle**, you might read "SHOP". If you tilt your head and look again from a **different angle**, you might read "STOP". If you combine both readings, you're more likely to get the correct answer than relying on just one look.

TTA does the same thing — it gives the model multiple "angles" on the same image and trusts the average.

---

## Why Does Averaging Work?

When the model looks at an image, it outputs a **vector of scores** (called "logits") — one score per class. For Fashion MNIST with 10 classes, it might look like:

```
Original image  → [2.1, 0.3, 5.8, 1.2, 0.3, -1.1, 0.8, 3.2, -0.2, 0.1]
                                  ^^^^ highest → predicts class 2
                                  
Flipped image   → [1.9, 0.5, 6.1, 0.8, 0.5, -0.9, 0.6, 3.5, -0.4, 0.3]
                                  ^^^^ highest → predicts class 2

Average         → [2.0, 0.4, 5.95, 1.0, 0.4, -1.0, 0.7, 3.35, -0.3, 0.2]
                                  ^^^^^ highest → predicts class 2 (more confident!)
```

In this case, both versions agreed, and the average made the model **more confident** in the right answer. But TTA really shines in **borderline cases** — images where the model is torn between two classes. The original might lean slightly wrong, but the flipped version might lean slightly right, and the average tips it to the correct class.

---

## TTA with Code — A Simple Example

Before looking at the notebook's implementation, let's see TTA in the simplest possible code:

```python
# Suppose we already have a trained model and a single test image
image = get_test_image()                  # shape: (1, 1, 28, 28)

# Step 1: Predict on the original image
scores_original = model(image)            # shape: (1, 10) — 10 class scores

# Step 2: Predict on the flipped version
flipped_image = torch.flip(image, [-1])   # flip horizontally (last dim = width)
scores_flipped = model(flipped_image)     # shape: (1, 10) — 10 class scores

# Step 3: Average the scores
averaged_scores = (scores_original + scores_flipped) / 2   # shape: (1, 10)

# Step 4: Pick the class with the highest averaged score
final_prediction = averaged_scores.argmax(dim=1)            # shape: (1,)
```

That's the entire concept. Everything in the notebook is just scaling this up to work on the **entire validation set** efficiently using callbacks.

---

## How TTA Is Implemented in This Notebook

The notebook implements TTA in two stages: **(A)** capture predictions, then **(B)** combine them. Let's walk through each part.

### Part A: Capturing Predictions — `CapturePreds` Callback

The model processes data in **batches** (e.g., 1024 images at a time). We need a way to collect predictions from every batch and stitch them together into one big tensor. That's what `CapturePreds` does:

```python
class CapturePreds(Callback):
    def before_fit(self, learn): 
        self.all_inps  = []    # will collect input images
        self.all_preds = []    # will collect model predictions
        self.all_targs = []    # will collect correct labels
    
    def after_batch(self, learn):
        self.all_inps.append(to_cpu(learn.batch[0]))     # input images
        self.all_preds.append(to_cpu(learn.preds))        # model's predictions
        self.all_targs.append(to_cpu(learn.batch[1]))     # correct labels
    
    def after_fit(self, learn):
        self.all_preds, self.all_targs, self.all_inps = map(
            torch.cat, [self.all_preds, self.all_targs, self.all_inps]
        )
```

**What's a Callback?** It's a class that "hooks into" the training/evaluation loop at specific moments. The Learner calls specific methods on each callback as it progresses:

| Method | When it runs | What we do here |
|--------|-------------|-----------------|
| `before_fit` | Once, before everything starts | Create empty lists to store results |
| `after_batch` | After each batch is processed | Append this batch's predictions, inputs, and labels to our lists |
| `after_fit` | Once, after everything finishes | Concatenate all the per-batch tensors into one big tensor each |

**Why `to_cpu()`?** During evaluation, data lives on the GPU for fast computation. But GPU memory is limited. By moving each batch's results to CPU immediately, we free up GPU memory for the next batch. CPU RAM is much larger.

**Why lists first, then `torch.cat`?** We don't know ahead of time how many total samples there are (and the last batch might be smaller than the rest). So we collect per-batch tensors in a list, then use `torch.cat` to glue them together:

```
Before torch.cat:  [tensor(1024, 10), tensor(1024, 10), ..., tensor(800, 10)]
After torch.cat:    tensor(10000, 10)   ← one big tensor for the whole validation set
```

### Part A (continued): The `capture_preds` Convenience Method

```python
@fc.patch
def capture_preds(self: Learner, cbs=None, inps=False):
    cp = CapturePreds()
    self.fit(1, train=False, cbs=[cp] + fc.L(cbs))
    res = cp.all_preds, cp.all_targs
    if inps: res = res + (cp.all_inps,)
    return res
```

This is a convenience wrapper that gets "patched" onto the `Learner` class (so we can call `learn.capture_preds()`). Here's what each part does:

- **`cp = CapturePreds()`** — create a fresh callback instance with empty lists
- **`self.fit(1, train=False, cbs=[cp] + fc.L(cbs))`** — this is the key line:
  - `1` — run for 1 epoch (one full pass through the validation data)
  - `train=False` — **evaluation only**, no weight updates, no gradients, BatchNorm in eval mode
  - `cbs=[cp] + fc.L(cbs)` — use our CapturePreds callback **plus** any extra callbacks we pass in (this is how we'll inject the flip transform for TTA). `fc.L(None)` safely becomes `[]`, so if we pass no extra callbacks, it just uses `[cp]`.
- **Returns** `(predictions, targets)` — or optionally `(predictions, targets, inputs)` if `inps=True`

### Part B: The Actual TTA — Getting Two Sets of Predictions and Combining Them

Now we use `capture_preds` twice — once on original images, once on flipped images:

```python
# Step 1: Predictions on ORIGINAL images (no augmentation)
ap1, at = learn.capture_preds()
# ap1 shape: (10000, 10) — 10 raw scores per image
# at  shape: (10000,)    — correct label per image
```

```python
# Step 2: Predictions on FLIPPED images
ttacb = BatchTransformCB(partial(tfm_batch, tfm_x=TF.hflip), on_val=True)
ap2, at = learn.capture_preds(cbs=[ttacb])
# ap2 shape: (10000, 10) — 10 raw scores per flipped image
```

Let's unpack that `ttacb` line piece by piece:

| Piece | What it does |
|-------|-------------|
| `TF.hflip` | `torchvision.transforms.functional.hflip` — flips an image horizontally (mirror) |
| `partial(tfm_batch, tfm_x=TF.hflip)` | Creates a function that flips the **images** in a batch but leaves the **labels** unchanged |
| `BatchTransformCB(..., on_val=True)` | A callback that applies this transform to every batch. **`on_val=True` is critical** — normally augmentation is turned off during validation, but for TTA we *want* to augment at validation time. That's the whole idea! |
| `learn.capture_preds(cbs=[ttacb])` | Runs evaluation with the flip callback injected — every image gets flipped before the model sees it |

### Deep Dive: What Exactly Is `BatchTransformCB`?

`BatchTransformCB` is defined in the miniai `learner` module (from an earlier notebook). It's a Callback whose job is simple: **intercept every batch of data and apply a transformation to it before the model sees it.** Here's what it looks like conceptually:

```python
class BatchTransformCB(Callback):
    def __init__(self, tfm, on_val=True):
        self.tfm = tfm          # the transform function to apply
        self.on_val = on_val    # whether to apply during validation too
    
    def before_batch(self, learn):
        # Only apply transform if we're training, OR if on_val=True
        if self.on_val or learn.training:
            learn.batch = self.tfm(learn.batch)
```

That's it — it's a very small class. The key things to understand:

**`self.tfm`** — stores whatever function you pass in. In our TTA case, this is `partial(tfm_batch, tfm_x=TF.hflip)` — a function that takes a batch `(images, labels)` and returns `(flipped_images, labels)`.

**`self.on_val`** — this boolean flag is the critical switch:
- **`on_val=False`** (the default for training augmentation): the transform is applied **only during training**. During validation, batches pass through untouched. This is what you want for regular data augmentation — you augment training data to prevent overfitting, but evaluate on clean validation data.
- **`on_val=True`** (what we use for TTA): the transform is applied **during validation too**. This is the whole point of TTA — we *intentionally* want to transform validation images so we can get predictions on those transformed versions.

**`before_batch`** — this method runs right before each batch is fed to the model. It replaces `learn.batch` with the transformed version. So the model never knows the difference — it just receives data and makes predictions, unaware that the images were flipped.

### The Full Chain: How `ttacb` Works End to End

Before we trace the steps, let's recall the definition of `tfm_batch` — a small but crucial helper function:

```python
def tfm_batch(b, tfm_x=fc.noop, tfm_y=fc.noop):
    return tfm_x(b[0]), tfm_y(b[1])
```

This function takes a batch `b` (which is a tuple of `(images, labels)`) and applies **separate** transforms to the images and labels. By default, both `tfm_x` and `tfm_y` are `fc.noop` (which means "no operation" — just return the input unchanged). The beauty of this design is that we can transform the images while leaving the labels alone, or vice versa, or both — it's completely flexible.

For TTA, we called `partial(tfm_batch, tfm_x=TF.hflip)`. This "bakes in" `TF.hflip` as the image transform, while `tfm_y` stays as the default `fc.noop`. So the resulting function will flip every image horizontally but leave every label untouched — exactly what we need, because flipping a T-shirt image doesn't change the fact that it's a T-shirt.

Now let's trace through exactly what happens when we run `learn.capture_preds(cbs=[ttacb])`:

**Step 1: A batch is loaded from the validation dataloader**

```
learn.batch = (images, labels)     # images shape: (1024, 1, 28, 28)
                                   # labels shape: (1024,)
```

The Learner pulls the next batch from the validation DataLoader. At this point, the images are completely normal, unmodified validation images — 1024 grayscale 28×28 images of clothing items. The labels are integers 0–9 telling us the correct class for each image.

**Step 2: `BatchTransformCB.before_batch` fires — this is where the flip happens**

The Learner calls `before_batch` on all active callbacks before the model sees the data. Our `BatchTransformCB` callback intercepts the batch and transforms it:

```
learn.batch = self.tfm(learn.batch)
```

Now `self.tfm` is the function we created earlier: `partial(tfm_batch, tfm_x=TF.hflip)`. Let's follow exactly what happens when it's called with `learn.batch`:

```python
# self.tfm is: partial(tfm_batch, tfm_x=TF.hflip)
# Calling it with learn.batch as the argument:

self.tfm(learn.batch)
# ↓ expands to:
partial(tfm_batch, tfm_x=TF.hflip)(learn.batch)
# ↓ partial fills in tfm_x, so this becomes:
tfm_batch(learn.batch, tfm_x=TF.hflip, tfm_y=fc.noop)
# ↓ inside tfm_batch, b = learn.batch = (images, labels):
#   b[0] = images (shape: 1024, 1, 28, 28)
#   b[1] = labels (shape: 1024,)
# ↓ the function body executes:
return TF.hflip(b[0]), fc.noop(b[1])
# ↓ TF.hflip flips all 1024 images horizontally
# ↓ fc.noop returns the labels completely unchanged
return (flipped_images, labels)
```

**The result:** `learn.batch` is now `(flipped_images, labels)`. Every image in the batch has been mirror-flipped (left↔right), but the labels are exactly the same. A T-shirt label of `0` is still `0` — because a flipped T-shirt is still a T-shirt.

This is why `tfm_batch` is so useful — it gives us a clean way to say "transform the images with this function, but don't touch the labels." Without it, we'd have to manually unpack the batch, apply the flip, and repack it. `tfm_batch` handles that plumbing for us.

**Step 3: The model makes predictions on the flipped images**

```
learn.preds = model(flipped_images)   # shape: (1024, 10)
```

The model has no idea the images were flipped — it just receives a tensor of images and outputs its predictions (10 raw scores per image, one per class). These predictions reflect what the model "thinks" when looking at the mirror version of each image. Some scores will be slightly different from what the model would have predicted on the original images, and that's exactly the variation we want to capture and average later.

**Step 4: `CapturePreds.after_batch` fires — collecting the results**

```python
self.all_inps.append(to_cpu(learn.batch[0]))    # store the flipped images
self.all_preds.append(to_cpu(learn.preds))       # store the predictions
self.all_targs.append(to_cpu(learn.batch[1]))    # store the labels
```

After the model has made its predictions, the `CapturePreds` callback grabs everything — the (flipped) input images, the model's predictions, and the correct labels — moves them to CPU to free GPU memory, and appends them to the growing lists. This batch's work is done.

**Step 5: Repeat for all batches...**

Steps 1–4 repeat for every batch in the validation set. With 10,000 validation images and a batch size of 1024, that's about 10 batches. Each batch goes through the exact same pipeline: load → flip → predict → capture.

**Step 6: `CapturePreds.after_fit` concatenates everything**

```python
self.all_preds, self.all_targs, self.all_inps = map(
    torch.cat, [self.all_preds, self.all_targs, self.all_inps]
)
```

After all batches are processed, the per-batch tensors are concatenated into single tensors covering the entire validation set:

```
ap2: (10000, 10) — predictions on ALL flipped images
at:  (10000,)    — all correct labels (unchanged by flipping)
```

These are the predictions the model made when it saw every validation image in its flipped form. Combined with `ap1` (predictions on original images), we now have two "opinions" per image that we can average together for TTA.

### Where Each Piece Fits in the Pipeline

Here's a visual summary of how `tfm_batch`, `partial`, `BatchTransformCB`, and `CapturePreds` each play their role:

```
tfm_batch          →  Generic batch transformer ("apply X to images, Y to labels")
partial(...)       →  Specializes tfm_batch for our use case ("apply hflip to images")
BatchTransformCB   →  Plugs the specialized function into the Learner's callback system
CapturePreds       →  Collects all the predictions after the model runs

Together they form a pipeline:

  DataLoader → BatchTransformCB → Model → CapturePreds → ap2
               (flips images)    (predicts)  (collects)
```

So `BatchTransformCB` sits between the data loader and the model, silently modifying every batch. It's a clean, modular design — the model code doesn't change, the data loading code doesn't change, we just plug in a callback that transforms the data in flight.

```python
# Step 3: Average and pick the winning class
ap = torch.stack([ap1, ap2]).mean(0).argmax(1)
```

This single line is the heart of TTA. Let's trace through it:

```
torch.stack([ap1, ap2])     →  shape: (2, 10000, 10)
                                       ^ two "versions" (original + flipped)

.mean(0)                    →  shape: (10000, 10)
                                       averages the two versions together
                                       for each image and each class

.argmax(1)                  →  shape: (10000,)
                                       picks the class with the highest 
                                       averaged score for each image
```

**Why `.argmax(1)` and not `.argmax(0)`?**

This is a great question, and the answer becomes clear once you think about what each dimension means in the tensor after `.mean(0)`:

```
After .mean(0), shape is: (10000, 10)
                            ↑       ↑
                          dim 0   dim 1
                         samples  classes
```

- **Dimension 0** = which image (0 to 9999). Each row is one image.
- **Dimension 1** = which class (0 to 9). Each column is the score for one class.

When we call `.argmax(1)`, we're saying: **"for each image (row), find which class (column) has the highest score."** That's exactly what we want — for each image, tell me the predicted class.

If we used `.argmax(0)` instead, we'd be asking: "for each class (column), which image (row) has the highest score?" That would give us a tensor of shape `(10,)` — telling us which image scored highest for each class. That's a completely different and useless question for classification!

Here's a concrete example to make it visual:

```python
# After .mean(0), imagine this 3×4 tensor (3 images, 4 classes):
#
#              class0  class1  class2  class3
# image 0:    [ 1.2,   3.5,    0.8,    2.1 ]
# image 1:    [ 0.5,   0.3,    4.2,    1.0 ]
# image 2:    [ 2.8,   1.1,    0.6,    3.9 ]

# .argmax(1) → for each IMAGE, which CLASS wins?
#   image 0 → class 1 (3.5 is highest)    ✓ This is what we want!
#   image 1 → class 2 (4.2 is highest)
#   image 2 → class 3 (3.9 is highest)
#   Result: tensor([1, 2, 3])  — shape (3,) — one predicted class per image

# .argmax(0) → for each CLASS, which IMAGE scores highest?
#   class 0 → image 2 (2.8 is highest)    ✗ Not useful for classification!
#   class 1 → image 0 (3.5 is highest)
#   class 2 → image 1 (4.2 is highest)
#   class 3 → image 2 (3.9 is highest)
#   Result: tensor([2, 0, 1, 2])  — shape (4,) — meaningless for predictions
```

**General rule of thumb:** in a `(samples, classes)` tensor, always use `.argmax(1)` (along the class dimension) to get predicted class labels.

**Why `torch.stack` and not `torch.cat`?**
- `torch.cat([ap1, ap2])` would give `(20000, 10)` — it would **concatenate** along the batch dimension, mixing the two sets together. That's not what we want.
- `torch.stack([ap1, ap2])` gives `(2, 10000, 10)` — it creates a **new dimension**, so each image has exactly 2 prediction vectors that we can average across.

```python
# Step 4: Calculate accuracy
tta_acc = round((ap == at).float().mean().item(), 3)
```

Breaking this down:
- `ap == at` → boolean tensor: `True` where prediction matches the correct label
- `.float()` → converts `True→1.0`, `False→0.0`
- `.mean()` → fraction of correct predictions = **accuracy**
- `.item()` → extracts the Python float from the tensor
- `round(..., 3)` → rounds to 3 decimal places

---

## The Full TTA Flow — Visual Summary

```
  Validation images (10000 images)
         │
         ├─────────────────────────────────────┐
         │                                     │
         ▼                                     ▼
  Original images                      Horizontally flipped images
         │                                     │
         ▼                                     ▼
     Model predicts                       Model predicts
     ap1: (10000, 10)                     ap2: (10000, 10)
         │                                     │
         └──────────────┬──────────────────────┘
                        │
                        ▼
              torch.stack([ap1, ap2])
                → (2, 10000, 10)
                        │
                        ▼
                    .mean(0)
            Average across the 2 versions
                → (10000, 10)
                        │
                        ▼
                   .argmax(1)
            Pick highest-scoring class
                → (10000,)
                        │
                        ▼
              Compare with targets (at)
              TTA Accuracy: ~0.5-1% better!
```

---

## Key Takeaways

1. **TTA is dead simple**: predict on original + predict on augmented versions + average the predictions.

2. **No retraining needed**: you use the exact same model with the exact same weights. You just feed it different versions of the same image.

3. **It's "free" accuracy**: typical improvement is 0.5–1% on top of your existing model. The only cost is inference time (2x the forward passes in this example).

4. **You can extend it**: use more augmentations (random crops, rotations, brightness changes) for more "opinions" to average. Each additional augmentation adds another forward pass.

```python
# Example: TTA with 4 augmented versions
ap1, at = learn.capture_preds()                              # original
ap2, at = learn.capture_preds(cbs=[flip_cb])                 # flipped
ap3, at = learn.capture_preds(cbs=[crop_cb])                 # cropped
ap4, at = learn.capture_preds(cbs=[flip_and_crop_cb])        # flipped + cropped

ap = torch.stack([ap1, ap2, ap3, ap4]).mean(0).argmax(1)     # average all 4
```

5. **Diminishing returns**: going from 1→2 versions helps a lot. Going from 4→5 helps less. Going from 20→21 barely matters. In practice, 2–5 augmented versions is the sweet spot.

6. **The key insight**: we average the **raw logits** (scores), not the final class predictions. Averaging `[0.51, 0.49]` and `[0.48, 0.52]` gives `[0.495, 0.505]` → class 1. But if we averaged the argmax predictions (class 0 and class 1), we'd have a tie. Averaging raw scores preserves the nuance that argmax throws away.

---

## Important Clarification: TTA Does NOT Help Training

This is a common point of confusion, so let's be very clear: **TTA has absolutely nothing to do with training. It is purely an inference/prediction-time technique.**

Here's the timeline of when TTA fits into the workflow:

```
PHASE 1: TRAINING (TTA not involved at all)
─────────────────────────────────────────────
  for each epoch:
      for each batch:
          predictions = model(batch)
          loss = loss_fn(predictions, targets)
          loss.backward()          ← gradients computed
          optimizer.step()         ← weights updated
  
  → Model weights are now FROZEN. Training is DONE.


PHASE 2: INFERENCE / DEPLOYMENT (where TTA lives)
─────────────────────────────────────────────
  The model's weights never change again.
  
  Without TTA:
      prediction = model(image)
      → single prediction, done
  
  With TTA:
      pred1 = model(original_image)
      pred2 = model(flipped_image)
      final = average(pred1, pred2)
      → better prediction, still done. No weights changed.
```

**TTA doesn't update any weights.** It doesn't call `loss.backward()`. It doesn't touch the optimizer. The model is completely frozen — we just run it multiple times on different versions of the same image and average the outputs.

So when the notebook calculates TTA accuracy, it's answering this question: **"If I use this trick at deployment time, how much better will my predictions be?"** It's measuring the improvement you'd get in production, not during training.

**Think of it this way:** training augmentation (RandomCrop, RandomFlip during training) and TTA are two sides of the same coin — both use image transformations, but at different stages and for different purposes:

| | Training Augmentation | Test Time Augmentation (TTA) |
|---|---|---|
| **When** | During training | After training is done |
| **Purpose** | Make the model learn better features | Make existing predictions more accurate |
| **Changes weights?** | Yes (indirectly, via better training) | No (model is frozen) |
| **Cost** | No extra cost (replaces original images) | 2–5x slower inference (multiple forward passes) |
| **Applied to** | Training data only (`on_val=False`) | Validation/test data (`on_val=True`) |
| **Permanent effect?** | Yes (model becomes better) | No (must re-apply every time you predict) |

In practice, you'd use **both**: training augmentation to train a better model, then TTA on top of that to squeeze out an extra 0.5–1% accuracy at inference time. They complement each other perfectly.

In [ ]:
# Visualize augmented images
xb, yb = learn.batch
show_images(xb[:16], imsize=1.5)

Notice how the images are shifted (RandomCrop) and some are flipped (RandomHorizontalFlip).

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. It loads ./interactive_viz/augmentation_playground.html
# at full width, sized to fit the visualization (setup cell near the top required).
# The full RandomCrop+Flip pipeline: pad → pick a corner (CLICK the padded
# image!) → crop → coin-flip. Press Play to stream endless variants of one image.
# ============================================================================
show_viz("interactive_viz/augmentation_playground.html", height="760px")

In [ ]:
#| export
@fc.patch
@fc.delegates(show_images)
def show_image_batch(self: Learner, max_n=9, cbs=None, **kwargs):
    """
    Show a batch of images from the learner.
    
    Useful for visualizing augmentations.
    
    Args:
        max_n: Maximum number of images to show (default: 9)
        cbs: Additional callbacks to use
        **kwargs: Additional arguments passed to show_images
    """
    self.fit(1, cbs=[SingleBatchCB()] + fc.L(cbs))
    show_images(self.batch[0][:max_n], **kwargs)

In [ ]:
# Use convenience method
learn.show_image_batch(max_n=16, imsize=1.5)

### Training with Augmentation

In [ ]:
# Use smaller padding (1 instead of 4) for more subtle augmentation
tfms = nn.Sequential(
    transforms.RandomCrop(28, padding=1),
    transforms.RandomHorizontalFlip()
)
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)

In [ ]:
# Train for more epochs with augmentation
set_seed(42)
epochs = 20
lr = 1e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched), augcb]  # Include augmentation callback

model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(epochs)

In [ ]:
# Save the trained model
mdl_path = Path('models')
mdl_path.mkdir(exist_ok=True)
torch.save(learn.model, mdl_path/'data_aug.pkl')

---
## Part 6: Test Time Augmentation (TTA)

**TTA** applies augmentation at inference time and averages predictions. This can improve accuracy without retraining!

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. It loads ./interactive_viz/tta_explorer.html
# at full width, sized to fit the visualization (setup cell near the top required).
# TTA step by step: predict original → predict flipped → stack & mean → argmax.
# Try the scenario where the original prediction is WRONG and TTA rescues it.
# ============================================================================
show_viz("interactive_viz/tta_explorer.html", height="740px")

In [ ]:
#| export
class CapturePreds(Callback):
    """
    Callback to capture predictions during validation.
    
    Stores all predictions, targets, and optionally inputs
    for later analysis.
    """
    def before_fit(self, learn): 
        # Initialize empty lists
        self.all_inps = []   # Inputs
        self.all_preds = []  # Predictions
        self.all_targs = []  # Targets
    
    def after_batch(self, learn):
        # Capture each batch (move to CPU to save GPU memory)
        self.all_inps.append(to_cpu(learn.batch[0]))
        self.all_preds.append(to_cpu(learn.preds))
        self.all_targs.append(to_cpu(learn.batch[1]))
    
    def after_fit(self, learn):
        # Concatenate all batches into single tensors
        self.all_preds, self.all_targs, self.all_inps = map(
            torch.cat, [self.all_preds, self.all_targs, self.all_inps]
        )

In [ ]:
#| export
@fc.patch
def capture_preds(self: Learner, cbs=None, inps=False):
    """
    Run validation and capture all predictions.
    
    Args:
        cbs: Additional callbacks to use
        inps: Whether to also return inputs (default: False)
    
    Returns:
        (predictions, targets) or (predictions, targets, inputs)
    """
    cp = CapturePreds()
    self.fit(1, train=False, cbs=[cp] + fc.L(cbs))
    res = cp.all_preds, cp.all_targs
    if inps: 
        res = res + (cp.all_inps,)
    return res

In [ ]:
# Get baseline predictions (no augmentation)
ap1, at = learn.capture_preds()

In [ ]:
# Get predictions with horizontal flip (TTA)
ttacb = BatchTransformCB(partial(tfm_batch, tfm_x=TF.hflip), on_val=True)
ap2, at = learn.capture_preds(cbs=[ttacb])

In [ ]:
# Check shapes
ap1.shape, ap2.shape, at.shape

In [ ]:
# Average predictions from both versions
# Stack predictions: (2, n_samples, n_classes)
# Mean over first dimension: (n_samples, n_classes)
# Argmax to get predicted class
ap = torch.stack([ap1, ap2]).mean(0).argmax(1)

In [ ]:
# Calculate TTA accuracy
tta_acc = round((ap == at).float().mean().item(), 3)
print(f"TTA Accuracy: {tta_acc}")

TTA typically improves accuracy by 0.5-1% without any retraining!

### TTA Code (Full) --Commented

In [ ]:
# =============================================================================
# TEST TIME AUGMENTATION (TTA) — FULLY EXPLAINED
# =============================================================================
#
# WHAT IS THIS FILE ABOUT?
# ------------------------
# This file implements Test Time Augmentation (TTA) — a technique to improve
# model accuracy at inference (prediction) time WITHOUT retraining the model.
#
# THE CORE IDEA:
# When a human looks at a photo, they can recognize the subject whether it's
# flipped, slightly cropped, or rotated. But a neural network might give
# slightly different confidence scores for "cat" vs "dog" depending on
# whether the image is flipped or not. TTA exploits this by:
#   1. Making predictions on the ORIGINAL image
#   2. Making predictions on AUGMENTED versions (e.g., horizontally flipped)
#   3. AVERAGING all the predictions together
# The average is usually more accurate than any single prediction, because
# random errors in individual predictions tend to cancel out.
#
# ANALOGY:
# Imagine asking 2 friends to identify a bird in a photo. If you also show
# them a flipped version of the photo, you now have 4 opinions instead of 2.
# The majority vote is more likely to be correct. TTA does the same thing
# but with a neural network looking at different versions of the same image.
#
# WHY DOES THIS WORK?
# Neural networks output a vector of "logits" (raw scores) for each class.
# For example, for Fashion MNIST with 10 classes:
#   Original image  → [2.1, 0.3, 5.8, 1.2, ...]  (class 2 = highest = "Pullover")
#   Flipped image   → [1.9, 0.5, 6.1, 0.8, ...]  (class 2 = still highest, but scores shifted)
#   Average         → [2.0, 0.4, 5.95, 1.0, ...]  (class 2 = even more confident)
# The averaging smooths out noise and makes the model more robust.
#
# PREREQUISITES:
# - A trained model (stored in a Learner object)
# - The Callback system from miniai (from previous notebooks in the course)
# - PyTorch basics (tensors, models, batches)
#
# =============================================================================


# =============================================================================
# IMPORTS (assumed to be available from earlier in the notebook)
# =============================================================================
# torch              — PyTorch, the deep learning framework
# fastcore.all as fc — fastcore library providing utilities like fc.patch, fc.L
# torch.cat          — Concatenates a list of tensors into one big tensor
# to_cpu             — Moves a tensor from GPU to CPU (saves GPU memory)
# Callback           — Base class for hooks that run at specific points during training
# Learner            — The training loop manager that holds model, data, loss, etc.
# BatchTransformCB   — A callback that applies a transformation to each batch
# partial            — From functools; lets you "pre-fill" some arguments of a function
# TF.hflip           — torchvision.transforms.functional.hflip: horizontally flips an image
# tfm_batch          — A helper function that applies separate transforms to x and y in a batch


# =============================================================================
# PART 1: CapturePreds — A Callback to Collect All Predictions
# =============================================================================
#
# WHAT IS A CALLBACK?
# -------------------
# In the miniai framework (built throughout the fast.ai Part 2 course), training
# happens inside a Learner's .fit() method, which runs a loop like:
#
#   for epoch in range(n_epochs):
#       for batch in dataloader:
#           predictions = model(batch_inputs)
#           loss = loss_function(predictions, batch_targets)
#           loss.backward()
#           optimizer.step()
#
# A "Callback" is an object that can inject custom code at specific points in
# this loop. The Learner calls methods on each callback at these moments:
#
#   before_fit()    → Called once before the entire training/validation run starts
#   before_epoch()  → Called at the start of each epoch
#   before_batch()  → Called before processing each batch
#   after_batch()   → Called after processing each batch (predictions are available)
#   after_epoch()   → Called at the end of each epoch
#   after_fit()     → Called once after the entire run finishes
#
# By subclassing Callback and overriding these methods, we can do anything
# we want at any point in the training loop — like collecting predictions!
#
# WHY DO WE NEED THIS?
# --------------------
# Normally during validation, the Learner computes metrics (like accuracy)
# on the fly and discards the raw predictions. But for TTA, we need to:
#   1. Run validation and KEEP all the raw predictions (logits)
#   2. Run validation AGAIN with augmented data and keep those predictions too
#   3. Average the predictions from step 1 and step 2
# CapturePreds lets us collect and store all predictions from a validation run.
#
# =============================================================================

#| export
# The #| export directive tells nbdev (the notebook→library tool) to include
# this code in the exported miniai/augment.py module. This is a fast.ai/nbdev
# convention — it means "this code is part of the library, not just a demo."

class CapturePreds(Callback):
    # CapturePreds inherits from Callback.
    # "Callback" is the base class defined in miniai/learner.py (from earlier notebooks).
    # By inheriting from it, we can override specific methods (before_fit, after_batch,
    # after_fit) to inject our prediction-capturing logic into the training loop.
    # Any method we DON'T override will use the default (do-nothing) behavior from
    # the base Callback class.

    def before_fit(self, learn):
        # =====================================================================
        # before_fit: Called ONCE before the validation loop begins.
        # =====================================================================
        #
        # WHAT IT DOES:
        # Initializes three empty Python lists to act as storage containers.
        # We'll fill these lists batch by batch during validation.
        #
        # WHY THREE LISTS?
        # We want to capture three things from each batch:
        #   1. all_inps  — the input images (useful for debugging/visualization)
        #   2. all_preds — the model's raw predictions (logits) for each image
        #   3. all_targs — the true labels (ground truth) for each image
        #
        # ARGUMENT:
        #   learn — the Learner object. We don't use it here, but the Callback
        #           protocol requires it. The Learner passes itself to every
        #           callback method so callbacks can access the model, data,
        #           optimizer, current batch, etc.
        #
        # WHY EMPTY LISTS (NOT TENSORS)?
        # During validation, data comes in batches (e.g., 1024 images at a time).
        # We don't know the total number of samples upfront. Lists are flexible —
        # we can .append() tensors of different batch sizes (the last batch may
        # be smaller). We'll concatenate them into one big tensor in after_fit().
        #
        self.all_inps = []      # Will hold: [batch0_inputs, batch1_inputs, ...]
        self.all_preds = []     # Will hold: [batch0_preds, batch1_preds, ...]
        self.all_targs = []     # Will hold: [batch0_targets, batch1_targets, ...]

    def after_batch(self, learn):
        # =====================================================================
        # after_batch: Called AFTER each batch is processed.
        # =====================================================================
        #
        # At this point in the Learner's loop, the following have already happened:
        #   1. A batch of data was loaded: learn.batch = (input_images, labels)
        #   2. The model made predictions: learn.preds = model(input_images)
        #   3. The loss was computed (but no backward pass during validation)
        #
        # WHAT IT DOES:
        # Grabs the inputs, predictions, and targets from the current batch
        # and appends them to our storage lists.
        #
        # ARGUMENT:
        #   learn — the Learner object, which gives us access to:
        #     learn.batch  — a tuple of (inputs, targets) for the current batch
        #                    learn.batch[0] = input images, shape (batch_size, 1, 28, 28)
        #                    learn.batch[1] = labels, shape (batch_size,)
        #     learn.preds  — model predictions (logits) for the current batch
        #                    shape (batch_size, 10) for 10-class Fashion MNIST
        #

        # --- Capture inputs ---
        # learn.batch[0] contains the input images for this batch.
        # to_cpu() moves the tensor from GPU memory to CPU memory.
        #
        # WHY to_cpu()?
        # During validation, data lives on the GPU (for fast model inference).
        # But GPU memory is limited and expensive. If we're collecting ALL
        # predictions across the entire validation set (10,000 images), we'd
        # use a lot of GPU memory for no reason. Moving to CPU frees GPU memory
        # so the next batch can use it. The CPU has much more RAM available.
        #
        # .append() adds this batch's tensor to the end of the list.
        # After all batches, self.all_inps might look like:
        #   [tensor(shape 1024,1,28,28), tensor(shape 1024,1,28,28), ..., tensor(shape 784,1,28,28)]
        # (last batch might be smaller if total samples isn't divisible by batch_size)
        #
        self.all_inps.append(to_cpu(learn.batch[0]))

        # --- Capture predictions ---
        # learn.preds contains the raw output from the model (logits).
        # These are NOT probabilities yet — they're raw scores. For example:
        #   [2.1, -0.5, 5.8, 1.2, 0.3, -1.1, 0.8, 3.2, -0.2, 0.1]
        # The class with the highest score is the predicted class.
        # We keep the RAW logits (not argmax) because TTA needs to AVERAGE
        # the logits from multiple augmented versions before taking argmax.
        #
        self.all_preds.append(to_cpu(learn.preds))

        # --- Capture targets (ground truth labels) ---
        # learn.batch[1] contains the true labels for this batch.
        # These are integer class indices: 0, 1, 2, ..., 9 for Fashion MNIST.
        # We need these to compute accuracy later: compare predictions vs targets.
        #
        self.all_targs.append(to_cpu(learn.batch[1]))

    def after_fit(self, learn):
        # =====================================================================
        # after_fit: Called ONCE after all batches have been processed.
        # =====================================================================
        #
        # WHAT IT DOES:
        # Concatenates the list of per-batch tensors into single large tensors.
        #
        # BEFORE this call, our lists look like:
        #   self.all_preds = [tensor(1024, 10), tensor(1024, 10), ..., tensor(784, 10)]
        #   self.all_targs = [tensor(1024,), tensor(1024,), ..., tensor(784,)]
        #   self.all_inps  = [tensor(1024,1,28,28), tensor(1024,1,28,28), ..., tensor(784,1,28,28)]
        #
        # AFTER this call, they become:
        #   self.all_preds = tensor(10000, 10)    — all predictions in one tensor
        #   self.all_targs = tensor(10000,)       — all targets in one tensor
        #   self.all_inps  = tensor(10000,1,28,28) — all inputs in one tensor
        #   (10000 = total number of validation samples in Fashion MNIST)
        #
        # HOW torch.cat WORKS:
        # torch.cat takes a list of tensors and joins them along dimension 0 (by default).
        # Example:
        #   torch.cat([tensor(3, 10), tensor(2, 10)]) → tensor(5, 10)
        # It stacks them vertically (along the batch dimension).
        #
        # HOW map() WORKS HERE:
        # map(function, iterable) applies the function to each element.
        # map(torch.cat, [list_of_pred_tensors, list_of_targ_tensors, list_of_inp_tensors])
        # is equivalent to:
        #   (torch.cat(list_of_pred_tensors), torch.cat(list_of_targ_tensors), torch.cat(list_of_inp_tensors))
        #
        # The result is a tuple of 3 concatenated tensors, which we unpack into
        # the same three attribute names, replacing the lists.
        #
        # WHY DO THIS?
        # Having one big tensor (instead of a list of small tensors) is much easier
        # to work with for downstream operations like computing accuracy, averaging
        # predictions, etc. You can index, slice, and do math on one tensor directly.
        #
        self.all_preds, self.all_targs, self.all_inps = map(
            torch.cat,                                           # Function to apply
            [self.all_preds, self.all_targs, self.all_inps]      # Three lists to concatenate
        )


# =============================================================================
# PART 2: capture_preds — Convenience Method on Learner
# =============================================================================
#
# WHAT IS @fc.patch?
# ------------------
# @fc.patch is a decorator from the fastcore library. It takes a standalone
# function and "patches" (attaches) it onto a class as a new method.
#
# So after this code runs:
#   learn = TrainLearner(...)
#   learn.capture_preds()     # ← This works! The function became a method.
#
# WHY USE @fc.patch INSTEAD OF DEFINING IT INSIDE THE CLASS?
# Because the Learner class was defined in a PREVIOUS notebook (miniai/learner.py).
# We don't want to go back and edit that file every time we add a new feature.
# @fc.patch lets us add methods to existing classes from anywhere. This is a
# key pattern in the fast.ai / nbdev workflow: build things incrementally
# across notebooks, adding methods to classes as needed.
#
# WHAT DOES THIS METHOD DO?
# It runs one epoch of validation (no training) with a CapturePreds callback
# attached, then returns the collected predictions and targets. It's a clean
# one-liner interface for "give me all the model's predictions on the validation set."
#
# =============================================================================

#| export
@fc.patch
def capture_preds(self: Learner, cbs=None, inps=False):
    # =========================================================================
    # capture_preds: Run validation and return all predictions.
    # =========================================================================
    #
    # ARGUMENTS:
    #
    #   self: Learner
    #     The Learner instance this method is called on (e.g., learn.capture_preds()).
    #     "self: Learner" in the function signature is how @fc.patch knows which
    #     class to attach this method to. After patching, 'self' works just like
    #     'self' in any normal class method — it refers to the Learner instance.
    #
    #   cbs: None or list of Callbacks (default: None)
    #     Additional callbacks to use during this validation run.
    #     This is how we pass in augmentation transforms for TTA.
    #     For example, to flip all images during validation:
    #       learn.capture_preds(cbs=[flip_callback])
    #     If None (default), no extra callbacks are added — just plain validation.
    #
    #   inps: bool (default: False)
    #     Whether to also return the input images alongside predictions and targets.
    #     Usually we only need predictions and targets (for computing accuracy).
    #     But sometimes we want the inputs too (for visualization/debugging):
    #       preds, targs, inputs = learn.capture_preds(inps=True)
    #     Default is False to save memory (inputs are large: images take up space).
    #

    # --- Step 1: Create a CapturePreds callback instance ---
    # This fresh instance has empty lists (initialized in before_fit).
    # It will fill up with predictions as validation batches are processed.
    cp = CapturePreds()

    # --- Step 2: Run one epoch of validation with the capture callback ---
    #
    # self.fit(1, train=False, cbs=[cp]+fc.L(cbs))
    #
    # Breaking this down piece by piece:
    #
    #   self.fit(1, ...)
    #     Run the training loop for 1 epoch. The Learner's .fit() method
    #     orchestrates the entire training/validation process.
    #
    #   train=False
    #     CRITICAL: This tells the Learner to SKIP the training phase entirely
    #     and ONLY run validation. We don't want to update the model's weights —
    #     we just want predictions. Setting train=False means:
    #       - The model is set to eval mode (model.eval())
    #       - No gradients are computed (saves memory and time)
    #       - Only the validation dataloader is iterated over
    #       - The optimizer does NOT update any weights
    #
    #   cbs=[cp] + fc.L(cbs)
    #     The list of callbacks to use for this run.
    #
    #     [cp] — Our CapturePreds callback (always included).
    #
    #     fc.L(cbs) — Wraps 'cbs' in a fastcore L (list-like) object.
    #       fc.L is a "smart list" from fastcore that handles None gracefully:
    #         fc.L(None)           → L([])       (empty list, not an error!)
    #         fc.L([callback1])    → L([callback1])
    #         fc.L(callback1)      → L([callback1])  (auto-wraps single items)
    #       This is important because cbs defaults to None. Without fc.L, we'd get:
    #         [cp] + None → TypeError!
    #       With fc.L:
    #         [cp] + fc.L(None) → [cp] + [] → [cp]  (works perfectly!)
    #
    #     So the full callback list is: [CapturePreds, ...any_extra_callbacks]
    #     For TTA, the extra callback might apply horizontal flips to each batch.
    #
    self.fit(1, train=False, cbs=[cp] + fc.L(cbs))

    # --- Step 3: Extract the captured results ---
    # After fit() completes, cp.after_fit() has already been called,
    # so cp.all_preds and cp.all_targs are now single concatenated tensors.
    #
    # res is a tuple: (all_predictions, all_targets)
    #   all_predictions shape: (n_validation_samples, n_classes) e.g., (10000, 10)
    #   all_targets shape:     (n_validation_samples,)           e.g., (10000,)
    #
    res = cp.all_preds, cp.all_targs

    # --- Step 4: Optionally include inputs ---
    # If inps=True, we append the input images to the result tuple.
    # This changes the return from (preds, targs) to (preds, targs, inputs).
    #
    # res + (cp.all_inps,) is tuple concatenation:
    #   (preds, targs) + (inputs,) → (preds, targs, inputs)
    #
    # Note the comma in (cp.all_inps,) — it's essential! Without it:
    #   (cp.all_inps) is just parentheses around a value, NOT a tuple.
    #   (cp.all_inps,) IS a tuple with one element.
    # Python requires the trailing comma to distinguish a single-element tuple
    # from a parenthesized expression.
    #
    if inps:
        res = res + (cp.all_inps,)

    # --- Step 5: Return the results ---
    # Returns either:
    #   (predictions_tensor, targets_tensor)                — if inps=False (default)
    #   (predictions_tensor, targets_tensor, inputs_tensor)  — if inps=True
    #
    return res


# =============================================================================
# PART 3: USING CAPTURE_PREDS AND TTA — STEP BY STEP
# =============================================================================

# ---- Step A: Get baseline predictions (no augmentation) ----
#
# learn.capture_preds() runs the validation set through the model ONCE
# with no modifications to the images. This gives us the "normal" predictions.
#
# ap1 = "augmented predictions 1" (but in this case, not actually augmented)
#   Shape: (10000, 10) — 10000 validation images, 10 class scores each
#   These are raw logits (unnormalized scores), NOT probabilities.
#   Example row: [2.1, -0.5, 5.8, 1.2, 0.3, -1.1, 0.8, 3.2, -0.2, 0.1]
#
# at = "all targets" — the ground truth labels
#   Shape: (10000,) — one integer label per image (0-9)
#
# We use tuple unpacking to assign both outputs at once:
#   ap1, at = learn.capture_preds()
# is equivalent to:
#   result = learn.capture_preds()
#   ap1 = result[0]
#   at = result[1]
#
ap1, at = learn.capture_preds()


# ---- Step B: Get predictions with horizontal flip (TTA) ----
#
# Now we run validation AGAIN, but this time every image is horizontally flipped
# before being fed to the model. The idea is that the model might give slightly
# different (sometimes better) scores when seeing the flipped version.
#
# Let's break down the callback creation:
#
#   TF.hflip
#     This is torchvision.transforms.functional.hflip — it flips an image
#     (or batch of images) horizontally (left ↔ right). Like looking in a mirror.
#     For Fashion MNIST: a T-shirt facing left becomes a T-shirt facing right.
#
#   partial(tfm_batch, tfm_x=TF.hflip)
#     partial() from functools creates a NEW function by "pre-filling" some arguments.
#
#     tfm_batch is defined as:
#       def tfm_batch(b, tfm_x=fc.noop, tfm_y=fc.noop):
#           return tfm_x(b[0]), tfm_y(b[1])
#     It takes a batch (images, labels) and applies separate transforms to each.
#
#     partial(tfm_batch, tfm_x=TF.hflip) creates a new function equivalent to:
#       def new_func(b, tfm_y=fc.noop):
#           return TF.hflip(b[0]), tfm_y(b[1])
#     So when this function receives a batch, it flips the images but leaves
#     the labels unchanged (fc.noop = "no operation" = identity function).
#
#   BatchTransformCB(partial(tfm_batch, tfm_x=TF.hflip), on_val=True)
#     BatchTransformCB is a callback that applies a given transform to every batch.
#
#     The first argument is the transform function to apply.
#
#     on_val=True is CRITICAL here!
#       Normally, augmentation callbacks have on_val=False (don't augment during
#       validation — you want to evaluate on clean data). But for TTA, we
#       INTENTIONALLY want to augment during validation. That's the whole point:
#       we want to see what the model predicts on flipped images.
#       Setting on_val=True means "yes, apply this transform during validation too."
#
ttacb = BatchTransformCB(partial(tfm_batch, tfm_x=TF.hflip), on_val=True)

# Now run validation with the flip callback:
#   learn.capture_preds(cbs=[ttacb])
# This runs fit(1, train=False, cbs=[CapturePreds(), ttacb])
# So every batch goes through: load → flip images → feed to model → capture predictions
#
# ap2 = predictions on flipped images
#   Shape: (10000, 10) — same shape as ap1
#
# at = targets (same as before — labels don't change when we flip images)
#   We overwrite 'at' but it's the same values since the targets are unchanged.
#
ap2, at = learn.capture_preds(cbs=[ttacb])


# ---- Step C: Average predictions and get final class predictions ----
#
# This is where the magic of TTA happens: we COMBINE the predictions from
# the original images and the flipped images.
#
# Let's trace through this line step by step:
#
#   torch.stack([ap1, ap2])
#     torch.stack takes a list of tensors with the SAME shape and stacks them
#     into a NEW dimension at position 0 (by default).
#
#     ap1 shape: (10000, 10)  — predictions on original images
#     ap2 shape: (10000, 10)  — predictions on flipped images
#     After stack: (2, 10000, 10)
#       Dimension 0 = which version (original=0, flipped=1)
#       Dimension 1 = which sample (0 to 9999)
#       Dimension 2 = which class (0 to 9)
#
#     Difference between stack and cat:
#       torch.cat([a, b])   → concatenates along existing dim: (20000, 10)
#       torch.stack([a, b]) → creates NEW dim: (2, 10000, 10)
#     We want stack because we want to average ACROSS versions, not mix samples.
#
#   .mean(0)
#     Takes the mean along dimension 0 (the "version" dimension).
#     For each of the 10000 samples, it averages the 2 prediction vectors.
#
#     Before mean: (2, 10000, 10)
#     After mean:  (10000, 10)
#
#     Example for one sample:
#       Original:  [2.1, -0.5, 5.8, 1.2, 0.3, -1.1, 0.8, 3.2, -0.2, 0.1]
#       Flipped:   [1.9, -0.3, 6.1, 0.8, 0.5, -0.9, 0.6, 3.5, -0.4, 0.3]
#       Average:   [2.0, -0.4, 5.95, 1.0, 0.4, -1.0, 0.7, 3.35, -0.3, 0.2]
#
#     WHY AVERAGE THE RAW LOGITS (not probabilities)?
#     Averaging logits works well in practice and is simpler than converting
#     to probabilities first (via softmax). Both approaches are used in the
#     literature, but logit averaging is the fast.ai convention. If we wanted
#     probability averaging, we'd do: torch.stack([ap1.softmax(1), ap2.softmax(1)]).mean(0)
#
#   .argmax(1)
#     Takes the index of the maximum value along dimension 1 (the "class" dimension).
#     This converts the averaged scores into a single predicted class per sample.
#
#     Before argmax: (10000, 10)  — 10 scores per sample
#     After argmax:  (10000,)    — 1 predicted class per sample
#
#     Example: [2.0, -0.4, 5.95, 1.0, 0.4, -1.0, 0.7, 3.35, -0.3, 0.2]
#              → argmax = 2 (because 5.95 is the highest value, at index 2)
#              → Predicted class: 2 (which is "Pullover" in Fashion MNIST)
#
#     WHY argmax AND NOT softmax?
#     We only need the PREDICTED CLASS (which class has the highest score),
#     not the actual probability. argmax gives us the index of the winner.
#     softmax would give probabilities but wouldn't change which class wins.
#
ap = torch.stack([ap1, ap2]).mean(0).argmax(1)
# ap shape: (10000,) — one predicted class index per validation image


# ---- Step D: Calculate TTA accuracy ----
#
# Now we compare the TTA predictions against the ground truth labels.
#
# Let's trace through this expression step by step:
#
#   (ap == at)
#     Element-wise comparison: is prediction equal to target?
#     ap shape: (10000,)  — predicted classes (integers 0-9)
#     at shape: (10000,)  — true classes (integers 0-9)
#     Result: (10000,) tensor of booleans (True where correct, False where wrong)
#     Example: tensor([True, True, False, True, False, ...])
#
#   .float()
#     Converts booleans to floats: True → 1.0, False → 0.0
#     This is necessary because .mean() doesn't work on boolean tensors.
#     Result: tensor([1.0, 1.0, 0.0, 1.0, 0.0, ...])
#
#   .mean()
#     Computes the mean of all values.
#     Since each value is 1.0 (correct) or 0.0 (incorrect),
#     the mean IS the accuracy (fraction of correct predictions).
#     Example: if 9340 out of 10000 are correct → mean = 0.934
#     Result: a scalar tensor, e.g., tensor(0.93400)
#
#   .item()
#     Extracts the Python float from a scalar tensor.
#     tensor(0.93400) → 0.934
#     This is needed because round() expects a Python number, not a tensor.
#     Also, plain Python floats are easier to print and work with.
#
#   round(..., 3)
#     Rounds to 3 decimal places.
#     0.93417... → 0.934
#     This makes the output clean and easy to read.
#
# The final result is a float like 0.934, representing 93.4% accuracy.
#
# TTA IMPROVEMENT:
# Without TTA (using ap1 alone): accuracy might be ~93.0%
# With TTA (averaging ap1 and ap2): accuracy might be ~93.5%
# That's a 0.5% improvement for FREE — no retraining needed!
# For larger models on harder tasks, TTA can give even bigger gains.
#
tta_accuracy = round((ap == at).float().mean().item(), 3)
# Example output: 0.934


# =============================================================================
# VISUAL SUMMARY OF THE ENTIRE TTA FLOW
# =============================================================================
#
#   Validation images (10000 images, 28x28 each)
#          │
#          ├──────────────────────────────────────────────┐
#          │                                              │
#          ▼                                              ▼
#   [Original images]                            [Horizontally flipped images]
#          │                                              │
#          ▼                                              ▼
#   Model predicts                                Model predicts
#   ap1: (10000, 10)                              ap2: (10000, 10)
#          │                                              │
#          └────────────────┬─────────────────────────────┘
#                           │
#                           ▼
#                 torch.stack([ap1, ap2])
#                   → (2, 10000, 10)
#                           │
#                           ▼
#                       .mean(0)
#                 Average across versions
#                   → (10000, 10)
#                           │
#                           ▼
#                      .argmax(1)
#                 Pick highest scoring class
#                   → (10000,)
#                           │
#                           ▼
#                 Compare with targets (at)
#                 Compute accuracy: 0.934
#
# =============================================================================
#
# EXTENDING TTA:
# You can add MORE augmented versions for even better results:
#   ap3 = predictions with random crop
#   ap4 = predictions with slight rotation
#   ap = torch.stack([ap1, ap2, ap3, ap4]).mean(0).argmax(1)
# More versions → better averaging → higher accuracy (with diminishing returns)
# The tradeoff is inference time: N versions = N times slower.
#
# =============================================================================

# Test Time Augmentation (TTA)

## The Problem: Our Model Only Sees One Version of Each Image

After training, when we evaluate our model on the test/validation set, we feed each image through the model **exactly once** and get a prediction. But think about this — if we slightly modify the image (say, flip it horizontally), the model might give a **slightly different prediction**. Sometimes that different prediction is actually **more correct**.

A T-shirt is still a T-shirt whether it's facing left or facing right. But our model might be slightly more confident about one orientation than the other, just because of the random patterns it happened to learn during training.

**What if we could get predictions from multiple versions of the same image and combine them?**

That's exactly what **Test Time Augmentation (TTA)** does.

## The Core Idea

TTA is beautifully simple:

1. Take the original test image → get predictions
2. Take a transformed version (e.g., horizontally flipped) → get predictions
3. **Average** the predictions together
4. Use the averaged prediction as the final answer

By averaging multiple "opinions" from the model, we smooth out individual mistakes. It's the same principle as asking 5 doctors instead of 1 — the consensus is usually more reliable than any single opinion.

And the best part? **We don't need to retrain anything.** TTA is completely free improvement at inference time (it just takes a bit longer since we run the model multiple times).

## Step 1: Building the Machinery to Capture Predictions

Before we can do TTA, we need a way to collect all predictions from a full pass through the validation set. That's what `CapturePreds` does.

```python
#| export
class CapturePreds(Callback):
    def before_fit(self, learn): self.all_inps,self.all_preds,self.all_targs = [],[],[]
    def after_batch(self, learn):
        self.all_inps. append(to_cpu(learn.batch[0]))
        self.all_preds.append(to_cpu(learn.preds))
        self.all_targs.append(to_cpu(learn.batch[1]))
    def after_fit(self, learn):
        self.all_preds,self.all_targs,self.all_inps = map(torch.cat, [self.all_preds,self.all_targs,self.all_inps])
```

This is a **Callback** — a class that "hooks into" the training/evaluation loop at specific moments and runs custom code. Our training loop processes data in batches (e.g., 1024 images at a time), so we need to collect predictions from every batch and stitch them together. Let me walk through each method:

### `before_fit` — Set Up Empty Collection Bins

```python
def before_fit(self, learn): self.all_inps,self.all_preds,self.all_targs = [],[],[]
```

This runs **once at the very beginning**, before any data is processed. It creates three empty lists that will serve as our collection bins:

- `self.all_inps` — will store all **input images** (the raw pixel data we feed into the model)
- `self.all_preds` — will store all **predictions** (what the model thinks each image is — raw scores for each of the 10 classes)
- `self.all_targs` — will store all **targets** (the correct labels — what each image actually is)

We start fresh every time so that predictions from a previous run don't contaminate the current one.

### `after_batch` — Collect Results from Each Batch

```python
def after_batch(self, learn):
    self.all_inps. append(to_cpu(learn.batch[0]))
    self.all_preds.append(to_cpu(learn.preds))
    self.all_targs.append(to_cpu(learn.batch[1]))
```

This runs **after every single batch** is processed. Let's break down each line:

**`learn.batch`** is a tuple of `(inputs, targets)` for the current batch:
- `learn.batch[0]` — the input images for this batch, shape `(1024, 1, 28, 28)` meaning 1024 grayscale 28×28 images
- `learn.batch[1]` — the correct labels for this batch, shape `(1024,)` meaning 1024 integers (each 0–9)

**`learn.preds`** — the model's predictions for this batch, shape `(1024, 10)` meaning 1024 sets of 10 class scores (one score per class, per image)

**`to_cpu(...)`** — moves the tensor from GPU back to CPU. We do this because:
1. GPU memory is limited and precious — we don't want to fill it up storing old predictions
2. We need the data on CPU anyway for our final analysis

**`.append(...)`** — adds each batch's results to our growing lists. After processing all batches, `self.all_preds` might look like `[tensor_of_1024_preds, tensor_of_1024_preds, ..., tensor_of_remaining_preds]`.

### `after_fit` — Stitch All Batches Together

```python
def after_fit(self, learn):
    self.all_preds,self.all_targs,self.all_inps = map(torch.cat, [self.all_preds,self.all_targs,self.all_inps])
```

This runs **once at the very end**, after all batches have been processed. Right now, each of our lists contains multiple tensors (one per batch). We need to combine them into single tensors for easy analysis.

**`torch.cat`** concatenates a list of tensors along the first dimension (stacking them vertically):
```
[tensor(1024, 10), tensor(1024, 10), ..., tensor(800, 10)]
    → one big tensor(10000, 10)
```

**`map(torch.cat, [list1, list2, list3])`** applies `torch.cat` to each of the three lists in one clean line. It's equivalent to writing:
```python
self.all_preds = torch.cat(self.all_preds)   # (10000, 10) — all predictions
self.all_targs = torch.cat(self.all_targs)   # (10000,)    — all correct labels
self.all_inps  = torch.cat(self.all_inps)    # (10000, 1, 28, 28) — all images
```

After this, we have complete tensors covering the **entire** validation set, not just individual batches.

## Step 2: A Convenience Method to Run Prediction Capture

```python
#| export
@fc.patch
def capture_preds(self: Learner, cbs=None, inps=False):
    cp = CapturePreds()
    self.fit(1, train=False, cbs=[cp]+fc.L(cbs))
    res = cp.all_preds,cp.all_targs
    if inps: res = res+(cp.all_inps,)
    return res
```

This adds a `capture_preds()` method to the `Learner` class using `@fc.patch` (which lets us add methods to existing classes without modifying their source code — the type hint `self: Learner` tells the decorator which class to patch).

Let me walk through each line:

**`cp = CapturePreds()`** — Creates a fresh instance of our callback that will collect predictions.

**`self.fit(1, train=False, cbs=[cp]+fc.L(cbs))`** — This is where the actual work happens. Let me unpack each argument:
- `1` — run for 1 epoch (one full pass through the validation data)
- `train=False` — evaluation mode only, no training. This means:
  - No gradient computation (saves memory and time)
  - BatchNorm uses its stored running statistics (not batch statistics)
  - Dropout is disabled (if any)
- `cbs=[cp]+fc.L(cbs)` — the callbacks to use during this run:
  - `cp` is our `CapturePreds` callback (always included)
  - `fc.L(cbs)` wraps `cbs` in a fastcore `L` list. This is a convenience — if `cbs` is `None`, `fc.L(None)` becomes an empty list `[]`, so `[cp]+[]` = `[cp]`. If `cbs` is a list of callbacks (like a transform callback for TTA), they get added too. This is how we'll inject the horizontal flip for TTA later.

**`res = cp.all_preds, cp.all_targs`** — After the evaluation run is complete, grab the predictions and targets from our callback. This gives us a tuple: `(predictions_tensor, targets_tensor)`.

**`if inps: res = res+(cp.all_inps,)`** — Optionally include the input images in the result. We usually don't need the inputs (they take a lot of memory), so this is `False` by default. But it's useful for debugging or visualization.

**`return res`** — Returns either `(predictions, targets)` or `(predictions, targets, inputs)`.

## Step 3: TTA in Action — Getting Two Sets of Predictions

Now we put it all together. First, we get predictions on the **original** (unmodified) validation images:

```python
ap1, at = learn.capture_preds()
```

- `ap1` — all predictions on original images, shape `(10000, 10)` — 10000 images, 10 class scores each
- `at` — all targets (correct labels), shape `(10000,)` — these don't change between runs

Next, we get predictions on **horizontally flipped** versions of the same images:

```python
ttacb = BatchTransformCB(partial(tfm_batch, tfm_x=TF.hflip), on_val=True)
ap2, at = learn.capture_preds(cbs=[ttacb])
```

Let me unpack this carefully:

**`TF.hflip`** — `torchvision.transforms.functional.hflip` — horizontally flips an image (mirror image, left becomes right). A T-shirt flipped horizontally is still a T-shirt.

**`partial(tfm_batch, tfm_x=TF.hflip)`** — creates a function that applies horizontal flip to the inputs (`tfm_x`) within a batch. `partial` pre-fills the `tfm_x` argument so we get a ready-to-use transform function.

**`BatchTransformCB(..., on_val=True)`** — a callback that applies a transformation to every batch:
- Normally, augmentations are applied only during **training** (to add variety and prevent overfitting)
- `on_val=True` is the critical part here — it tells the callback to **also apply this transform during validation/evaluation**. This is what makes it "test time" augmentation. Without `on_val=True`, the flip would be ignored during our evaluation run, and we'd just get the same predictions as `ap1`.

**`learn.capture_preds(cbs=[ttacb])`** — runs evaluation with our flip callback injected. Every batch of images gets horizontally flipped before being fed to the model. The model sees mirror images and gives its predictions on those flipped versions.

- `ap2` — all predictions on flipped images, shape `(10000, 10)`
- `at` — same targets as before (the correct answer doesn't change when we flip the image)

## Step 4: Combine the Predictions

```python
ap = torch.stack([ap1, ap2]).mean(0).argmax(1)
```

This single line is where the magic of TTA happens. Let me break it down step by step:

**`torch.stack([ap1, ap2])`** — stacks the two prediction tensors into a new dimension:
```
ap1 shape: (10000, 10)   — predictions from original images
ap2 shape: (10000, 10)   — predictions from flipped images

stacked shape: (2, 10000, 10)
                ^
                new dimension (2 "versions" of predictions)
```

**`.mean(0)`** — averages along dimension 0 (the "versions" dimension):
```
(2, 10000, 10) → (10000, 10)

For each image, for each class, we now have:
    averaged_score = (original_score + flipped_score) / 2
```

This is the ensemble step. If the original image gave class 3 a score of 0.7 and class 5 a score of 0.2, but the flipped image gave class 3 a score of 0.6 and class 5 a score of 0.3, the averaged scores would be class 3: 0.65, class 5: 0.25. The averaging smooths out noise and often corrects borderline mistakes.

**`.argmax(1)`** — for each image, find which class has the highest averaged score:
```
(10000, 10) → (10000,)

For each of the 10000 images, returns the INDEX (0-9) of the class
with the highest averaged score. This is the final predicted class.
```

## Step 5: Measure the Improvement

```python
round((ap==at).float().mean().item(), 3)
```

This calculates the accuracy of our TTA predictions:

**`ap == at`** — element-wise comparison: for each of the 10000 images, is our prediction correct? Returns a boolean tensor like `[True, True, False, True, ...]`

**`.float()`** — converts `True→1.0` and `False→0.0`, giving us `[1.0, 1.0, 0.0, 1.0, ...]`

**`.mean()`** — averages all the 1s and 0s. This IS the accuracy. For example, if 9200 out of 10000 are correct: `mean = 9200/10000 = 0.92`

**`.item()`** — extracts the single number from a 0-dimensional tensor (converts `tensor(0.92)` to plain Python float `0.92`)

**`round(..., 3)`** — rounds to 3 decimal places for a clean display (e.g., `0.923`)

## The Big Picture: Why TTA Works

```
                    Original Image          Flipped Image
                    ┌──────────┐            ┌──────────┐
                    │  👕      │            │      👕  │
                    │  (left)  │            │  (right) │
                    └────┬─────┘            └────┬─────┘
                         │                       │
                         ▼                       ▼
                    ┌──────────┐            ┌──────────┐
                    │  Model   │            │  Model   │
                    │ (same    │            │ (same    │
                    │ weights) │            │ weights) │
                    └────┬─────┘            └────┬─────┘
                         │                       │
                         ▼                       ▼
                   Predictions ap1          Predictions ap2
                   [0.1, 0.8, 0.1]         [0.05, 0.85, 0.1]
                         │                       │
                         └───────┬───────────────┘
                                 │
                                 ▼
                            Average them
                         [0.075, 0.825, 0.1]
                                 │
                                 ▼
                            argmax → Class 1 ✓
```

The model is the same — we don't change any weights. We just give it different "views" of the same image and trust that the **consensus** is more reliable than any single view.

This idea can be extended beyond just horizontal flips. We could also try:
- Small rotations
- Slight zoom-in/zoom-out
- Brightness/contrast adjustments
- Combinations of multiple augmentations

Each additional augmentation adds another "opinion" to average over, potentially improving accuracy further — though with diminishing returns and increasing compute cost.

---
## Part 7: Random Erase

**Random Erase** (also called Cutout) randomly erases rectangular regions of the image. This forces the model to look at the whole image, not just the most obvious features.

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. It loads ./interactive_viz/random_erase_lab.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Random Erase, one stage per line of _rand_erase1: size → position → gaussian
# fill → clamp → repeat. CLICK the image to place the patch; watch mean/std stay put.
# ============================================================================
show_viz("interactive_viz/random_erase_lab.html", height="760px")

In [ ]:
# Get a batch of images
xb, _ = next(iter(dls.train))
xbt = xb[:16]

In [ ]:
# Get stats for replacing erased region
xm, xs = xbt.mean(), xbt.std()

In [ ]:
# Check min/max values
xbt.min(), xbt.max()

In [ ]:
# Percentage of image to erase
pct = 0.2  # 20%

In [ ]:
# Calculate random position and size for erasing
szx = int(pct * xbt.shape[-2])  # Height of erased region
szy = int(pct * xbt.shape[-1])  # Width of erased region
stx = int(random.random() * (1-pct) * xbt.shape[-2])  # Start row
sty = int(random.random() * (1-pct) * xbt.shape[-1])  # Start column

print(f"Erase region: start=({stx},{sty}), size=({szx},{szy})")

In [ ]:
# Fill erased region with random noise (same mean/std as image)
init.normal_(xbt[:, :, stx:stx+szx, sty:sty+szy], mean=xm, std=xs);

In [ ]:
# Visualize
show_images(xbt, imsize=1.5)

In [ ]:
#|export
def _rand_erase1(x, pct, xm, xs, mn, mx):
    """
    Randomly erase a single rectangular region.
    
    Args:
        x: Image tensor
        pct: Percentage of image to erase (0-1)
        xm, xs: Mean and std for replacement noise
        mn, mx: Min and max values to clamp to
    """
    # Calculate region size
    szx = int(pct * x.shape[-2])
    szy = int(pct * x.shape[-1])
    
    # Random position (ensuring region fits within image)
    stx = int(random.random() * (1-pct) * x.shape[-2])
    sty = int(random.random() * (1-pct) * x.shape[-1])
    
    # Fill with random noise
    init.normal_(x[:, :, stx:stx+szx, sty:sty+szy], mean=xm, std=xs)
    
    # Clamp to valid range
    x.clamp_(mn, mx)

In [ ]:
# Test single erase
xb, _ = next(iter(dls.train))
xbt = xb[:16]
_rand_erase1(xbt, 0.2, xbt.mean(), xbt.std(), xbt.min(), xbt.max())
show_images(xbt, imsize=1.5)

In [ ]:
#|export
def rand_erase(x, pct=0.2, max_num=4):
    """
    Randomly erase 0 to max_num rectangular regions.
    
    Args:
        x: Image tensor
        pct: Percentage of image for each erased region
        max_num: Maximum number of regions to erase
    
    Returns:
        Modified image tensor
    """
    xm, xs, mn, mx = x.mean(), x.std(), x.min(), x.max()
    # Random number of regions (0 to max_num)
    num = random.randint(0, max_num)
    for i in range(num): 
        _rand_erase1(x, pct, xm, xs, mn, mx)
    return x

In [ ]:
# Test multiple erases
xb, _ = next(iter(dls.train))
xbt = xb[:16]
rand_erase(xbt, 0.2, 4)
show_images(xbt, imsize=1.5)

In [ ]:
#|export
class RandErase(nn.Module):
    """
    Random Erase as a PyTorch module.
    
    Can be used in nn.Sequential for easy composition.
    
    Args:
        pct: Percentage of image for each erased region (default: 0.2)
        max_num: Maximum number of regions to erase (default: 4)
    """
    def __init__(self, pct=0.2, max_num=4):
        super().__init__()
        self.pct, self.max_num = pct, max_num
    
    def forward(self, x): 
        return rand_erase(x, self.pct, self.max_num)

In [ ]:
# Create augmentation pipeline with RandErase
tfms = nn.Sequential(
    transforms.RandomCrop(28, padding=1),
    transforms.RandomHorizontalFlip(),
    RandErase()
)
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)

In [ ]:
# Visualize augmented batch
model = get_model()
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=[DeviceCB(), SingleBatchCB(), augcb])
learn.fit(1)
xb, yb = learn.batch
show_images(xb[:16], imsize=1.5)

### Random Erase Code (Full) - Commented

In [ ]:
# =============================================================================
# RANDOM ERASE (CUTOUT) — DATA AUGMENTATION TECHNIQUE — FULLY EXPLAINED
# =============================================================================
#
# WHAT IS THIS FILE ABOUT?
# ------------------------
# This file implements "Random Erase" (also known as "Cutout") — a data
# augmentation technique that randomly erases rectangular regions of training
# images and replaces them with random noise. This forces the neural network
# to learn from ALL parts of the image, not just the most obvious features.
#
# WHY IS RANDOM ERASE USEFUL?
# ----------------------------
# Imagine you're training a model to recognize T-shirts. Without Random Erase,
# the model might learn to look ONLY at the collar area (the most distinctive
# feature). If the collar is obscured in a test image, the model fails.
#
# With Random Erase, sometimes the collar gets erased during training, forcing
# the model to also learn from sleeves, hemlines, fabric texture, etc. This
# makes the model more robust and better at generalizing to new images.
#
# ANALOGY:
# It's like a teacher covering part of a flashcard during a quiz — the student
# has to learn ALL features of the subject, not just rely on one obvious clue.
#
# RESEARCH BACKGROUND:
# - "Random Erasing Data Augmentation" (Zhong et al., 2020)
# - "Cutout" (DeVries & Taylor, 2017) — similar technique
# Both papers show consistent accuracy improvements across many tasks.
#
# THIS FILE COVERS:
# 1. Step-by-step manual Random Erase to understand the mechanics
# 2. _rand_erase1() — erase a single random rectangle
# 3. rand_erase() — erase multiple random rectangles
# 4. RandErase — PyTorch module wrapper for use in augmentation pipelines
# 5. Integration into a training pipeline with other augmentations
#
# PREREQUISITES:
# - PyTorch tensors and basic indexing/slicing
# - Understanding of image tensors: shape (batch, channels, height, width)
# - The miniai framework (Learner, Callbacks, DataLoaders)
# - Basic understanding of data augmentation (from Part 5 of the notebook)
#
# =============================================================================


# =============================================================================
# IMPORTS (assumed to be available from earlier in the notebook)
# =============================================================================
# torch          — PyTorch deep learning framework
# random         — Python's built-in random number generator
# nn             — torch.nn: neural network modules
# init           — torch.nn.init: weight/tensor initialization functions
# dls            — DataLoaders object holding train/validation dataloaders
# show_images    — Utility function to display a grid of image tensors
# transforms     — torchvision.transforms: image transformation functions
# BatchTransformCB — Callback that applies a transform to each batch
# partial        — functools.partial: pre-fill function arguments
# tfm_batch      — Helper to apply separate transforms to images and labels
# get_model      — Function to create a ResNet model (from earlier in notebook)
# TrainLearner   — Extended Learner with training capabilities
# F              — torch.nn.functional: loss functions, etc.
# DeviceCB       — Callback that moves data to GPU
# SingleBatchCB  — Callback that stops training after one batch (for testing)


# =============================================================================
# PART 1: MANUAL RANDOM ERASE — UNDERSTANDING THE MECHANICS
# =============================================================================
#
# Before building reusable functions, let's manually walk through every step
# of erasing a random rectangle from a batch of images. This helps us
# understand exactly what's happening before we abstract it into functions.
#
# =============================================================================

# ---- Step 1: Get a batch of images from the training set ----
#
# dls.train is the training DataLoader. It yields batches of (images, labels).
#
# iter(dls.train)
#   Creates an "iterator" over the training DataLoader. An iterator is an object
#   that produces items one at a time. Think of it like pressing "next" on a
#   playlist — each press gives you the next song (batch).
#
#   WHY iter()?
#   A DataLoader by itself is "iterable" (you can loop over it), but to get
#   just ONE batch without a for-loop, we need to convert it to an iterator
#   first, then call next() on it.
#
# next(...)
#   Gets the FIRST item from the iterator — i.e., the first batch.
#   Each batch is a tuple: (images_tensor, labels_tensor)
#     images_tensor shape: (batch_size, 1, 28, 28)
#       batch_size = 1024 (configured earlier in notebook)
#       1 = number of color channels (grayscale; RGB would be 3)
#       28 x 28 = image height x width (Fashion MNIST is 28x28 pixels)
#     labels_tensor shape: (batch_size,)
#       Integer class labels: 0-9 for Fashion MNIST
#       (0=T-shirt, 1=Trouser, 2=Pullover, 3=Dress, 4=Coat,
#        5=Sandal, 6=Shirt, 7=Sneaker, 8=Bag, 9=Ankle boot)
#
# xb, _ = ...
#   Tuple unpacking: xb gets the images, _ gets the labels.
#   The underscore _ is a Python convention meaning "I don't need this value."
#   We only care about the images for this augmentation demo, not the labels.
#   This is a common pattern when you want to discard part of a returned tuple.
#
xb, _ = next(iter(dls.train))

# xbt = xb[:16]
#   Slicing: take only the FIRST 16 images from the batch of 1024.
#
#   WHY only 16?
#   We're about to VISUALIZE these images, and 16 fits nicely in a 4x4 grid.
#   Using all 1024 images would be impossible to display and wasteful for a demo.
#
#   xb[:16] uses Python slice notation:
#     xb[start:stop] where start defaults to 0
#     So xb[:16] = xb[0:16] = first 16 elements along dimension 0 (batch dim)
#
#   xbt shape: (16, 1, 28, 28) — 16 grayscale 28x28 images
#
xbt = xb[:16]


# ---- Step 2: Compute image statistics ----
#
# We need the mean and standard deviation of the image pixel values.
# These will be used later when we fill the erased region with random noise.
#
# xbt.mean()
#   Computes the mean of ALL values in the tensor (all pixels, all images).
#   Returns a single scalar tensor, e.g., tensor(-0.21)
#   (Negative because images were normalized: (pixel - 0.28) / 0.35)
#
# xbt.std()
#   Computes the standard deviation of ALL values in the tensor.
#   Returns a single scalar tensor, e.g., tensor(1.02)
#
# WHY DO WE NEED THESE STATISTICS?
# When we erase a region, we need to fill it with SOMETHING. Options:
#   1. Fill with zeros → creates an obvious black rectangle (bad — model
#      might learn "black rectangle = augmented" instead of learning features)
#   2. Fill with the mean value → gray rectangle, still conspicuous
#   3. Fill with random noise matching the image statistics → BEST OPTION
#      The noise looks "natural" and doesn't introduce a systematic pattern
#      that the model could learn to exploit.
#
# By using the same mean and std as the actual image data, the random noise
# will have similar brightness and contrast, making the erased region less
# "artificial" and harder for the model to detect/ignore.
#
xm, xs = xbt.mean(), xbt.std()


# ---- Step 3: Define how much of the image to erase ----
#
# pct = 0.2 means each erased rectangle will cover approximately 20% of
# the image along each dimension (height and width).
#
# The actual erased AREA is roughly pct * pct = 0.04 = 4% of total pixels.
# Wait — that seems small! But remember: we'll erase MULTIPLE rectangles,
# and the goal isn't to remove most of the image but to remove just enough
# to prevent the model from relying on any single local feature.
#
# CHOOSING pct:
# - Too small (e.g., 0.05): Barely noticeable, little effect
# - Too large (e.g., 0.5): Removes too much info, model can't learn
# - 0.2 is a good default that works well in practice
#
pct = 0.2  # 20%


# ---- Step 4: Calculate the position and size of the rectangle to erase ----

# --- Calculate the SIZE of the erased rectangle ---
#
# szx = int(pct * xbt.shape[-2])
#
#   xbt.shape = (16, 1, 28, 28)
#     xbt.shape[-2] = 28  (second-to-last dimension = height)
#     xbt.shape[-1] = 28  (last dimension = width)
#
#   WHY NEGATIVE INDEXING (-2, -1)?
#   Negative indices count from the END of the shape tuple:
#     shape[-1] = last element = width
#     shape[-2] = second-to-last = height
#   This is more robust than using shape[2] and shape[3], because it works
#   regardless of whether the tensor has a batch dimension or not.
#   (A single image might be shape (1, 28, 28) while a batch is (16, 1, 28, 28))
#
#   pct * xbt.shape[-2] = 0.2 * 28 = 5.6
#   int(5.6) = 5   (int() truncates toward zero, dropping the decimal)
#
#   So szx = 5: the erased rectangle is 5 pixels tall.
#
szx = int(pct * xbt.shape[-2])  # Height of erased region: int(0.2 * 28) = 5

# szy = int(pct * xbt.shape[-1])
#   Same calculation for width.
#   0.2 * 28 = 5.6 → int(5.6) = 5
#   So szy = 5: the erased rectangle is 5 pixels wide.
#
#   The erased rectangle is 5x5 = 25 pixels out of 28x28 = 784 total pixels
#   That's about 3.2% of the image area.
#
szy = int(pct * xbt.shape[-1])  # Width of erased region: int(0.2 * 28) = 5


# --- Calculate the POSITION (top-left corner) of the erased rectangle ---
#
# We want a RANDOM position, but we need to make sure the entire rectangle
# fits within the image boundaries (doesn't go off the edge).
#
# stx = int(random.random() * (1-pct) * xbt.shape[-2])
#
#   random.random()
#     Returns a random float between 0.0 (inclusive) and 1.0 (exclusive).
#     For example: 0.4372
#
#   (1-pct)
#     = 1 - 0.2 = 0.8
#     This factor ensures the rectangle fits within the image.
#
#   (1-pct) * xbt.shape[-2]
#     = 0.8 * 28 = 22.4
#     This is the maximum starting row. If stx goes up to 22,
#     and the rectangle is 5 pixels tall, it ends at row 22+5=27,
#     which is within the image (rows 0-27 for a 28-pixel-tall image).
#
#   random.random() * 22.4
#     A random value between 0 and 22.4
#     For example: 0.4372 * 22.4 = 9.79
#
#   int(9.79) = 9
#     So the rectangle starts at row 9 and extends to row 9+5=14.
#
#   WHY (1-pct) AND NOT JUST (shape - szx)?
#   Both would work! (1-pct) * shape ≈ shape - szx because szx = pct * shape.
#   The code uses (1-pct) for simplicity — it avoids referencing szx.
#   Mathematically: (1-pct) * H = H - pct*H ≈ H - szx
#
stx = int(random.random() * (1 - pct) * xbt.shape[-2])  # Random start row

# sty: Same logic for the column (horizontal position).
sty = int(random.random() * (1 - pct) * xbt.shape[-1])  # Random start column

# Print the computed region for inspection
# f-string formatting: f"..." allows embedding {variables} directly in strings
print(f"Erase region: start=({stx},{sty}), size=({szx},{szy})")
# Example output: "Erase region: start=(9,15), size=(5,5)"
# This means: erase a 5x5 rectangle starting at row 9, column 15.


# ---- Step 5: Fill the erased region with random noise ----
#
# init.normal_(xbt[:, :, stx:stx+szx, sty:sty+szy], mean=xm, std=xs)
#
# This is the actual "erasing" step. Let's break it down:
#
# xbt[:, :, stx:stx+szx, sty:sty+szy]
#   This is 4D tensor SLICING. It selects a rectangular region from the images.
#
#   xbt has shape (16, 1, 28, 28):
#     Dimension 0 (batch):    [:] = ALL 16 images (we erase the SAME region
#                              in every image — this is the "batch" approach)
#     Dimension 1 (channels): [:] = ALL channels (just 1 for grayscale)
#     Dimension 2 (height):   [stx:stx+szx] = rows from stx to stx+szx-1
#                              e.g., [9:14] = rows 9, 10, 11, 12, 13 (5 rows)
#     Dimension 3 (width):    [sty:sty+szy] = cols from sty to sty+szy-1
#                              e.g., [15:20] = cols 15, 16, 17, 18, 19 (5 cols)
#
#   Result shape: (16, 1, 5, 5) — the 5x5 region in all 16 images
#
#   IMPORTANT: This slice is a VIEW, not a copy. Modifying it modifies
#   the original tensor xbt in-place. This is how we "erase" the region.
#
# init.normal_(...)
#   torch.nn.init.normal_ fills a tensor with random values drawn from a
#   normal (Gaussian) distribution with the specified mean and std.
#
#   The underscore _ at the end means IN-PLACE: it modifies the tensor
#   directly rather than creating a new one. This is a PyTorch convention:
#     tensor.add_(5)    — adds 5 in-place (modifies tensor)
#     tensor.add(5)     — returns a new tensor (original unchanged)
#     init.normal_(t)   — fills t in-place with random normal values
#
#   mean=xm, std=xs
#     We use the image batch's own mean and standard deviation.
#     This makes the noise look "natural" — it has the same overall
#     brightness and contrast as the rest of the image.
#
#   WHAT HAPPENS VISUALLY:
#   Each pixel in the 5x5 region gets replaced by a random value drawn
#   from N(xm, xs²). The result looks like TV static/grain in that region.
#
#   The semicolon ; at the end suppresses the output in Jupyter notebooks.
#   Without it, Jupyter would print the return value of init.normal_()
#   (which is the tensor itself — not useful to see here).
#
init.normal_(xbt[:, :, stx:stx + szx, sty:sty + szy], mean=xm, std=xs);


# ---- Step 6: Visualize the erased images ----
#
# show_images(xbt, imsize=1.5)
#   Displays the 16 images in a grid.
#
#   xbt: the tensor of 16 images (now with erased rectangles)
#   imsize=1.5: each image is displayed at 1.5 inches
#
#   You'll see 16 Fashion MNIST images, each with a small noisy/grainy
#   rectangle at the SAME position. This is because we used the same
#   stx, sty for all images in the batch.
#
show_images(xbt, imsize=1.5)


# =============================================================================
# PART 2: _rand_erase1 — FUNCTION TO ERASE A SINGLE RANDOM RECTANGLE
# =============================================================================
#
# Now we wrap the manual steps above into a reusable function.
# The leading underscore _ in the name is a Python convention meaning
# "this is a private/internal helper function, not meant to be called
# directly by users." It will be called by the public rand_erase() function.
#
# =============================================================================

#|export
# #|export tells nbdev to include this function in miniai/augment.py

def _rand_erase1(x, pct, xm, xs, mn, mx):
    """
    Randomly erase a single rectangular region from a batch of images.

    This function picks a random position within the image, selects a rectangle
    whose size is proportional to 'pct' of the image dimensions, fills that
    rectangle with random Gaussian noise (matching the image statistics),
    and clamps all pixel values to stay within valid bounds.

    Args:
        x:   Image tensor, shape (batch, channels, height, width).
             All images in the batch get the same region erased.
        pct: Float between 0 and 1. Fraction of image dimension for the
             rectangle size. E.g., 0.2 means the rectangle is 20% of the
             image height and 20% of the image width.
        xm:  Mean of the image data (scalar tensor or float). Used as the
             mean of the Gaussian noise that fills the erased region.
        xs:  Standard deviation of the image data. Used as the std of the
             Gaussian noise.
        mn:  Minimum valid pixel value (scalar). Used for clamping.
        mx:  Maximum valid pixel value (scalar). Used for clamping.

    Returns:
        None — modifies x IN-PLACE (the original tensor is changed directly).
    """

    # --- Calculate the SIZE of the erased rectangle ---
    #
    # szx = height of rectangle = pct * image_height, truncated to integer
    # szy = width of rectangle  = pct * image_width, truncated to integer
    #
    # x.shape[-2] = image height (second-to-last dimension)
    # x.shape[-1] = image width (last dimension)
    #
    # For Fashion MNIST: int(0.2 * 28) = int(5.6) = 5 pixels
    #
    szx = int(pct * x.shape[-2])
    szy = int(pct * x.shape[-1])

    # --- Calculate the random POSITION (top-left corner) ---
    #
    # random.random() gives a uniform random float in [0, 1).
    #
    # Multiply by (1-pct) * dimension to ensure the rectangle fits:
    #   Maximum start position = (1-pct) * dimension
    #   Rectangle end = start + size ≤ (1-pct)*dim + pct*dim = dim  ✓
    #
    # For Fashion MNIST: int(random * 0.8 * 28) = int(random * 22.4)
    #   → a random integer from 0 to 22 (ensuring 5-pixel rect fits in 28 pixels)
    #
    stx = int(random.random() * (1 - pct) * x.shape[-2])  # Start row
    sty = int(random.random() * (1 - pct) * x.shape[-1])  # Start column

    # --- Fill the rectangle with random Gaussian noise ---
    #
    # x[:, :, stx:stx+szx, sty:sty+szy] selects the rectangle from ALL images
    # in the batch and ALL channels (see detailed explanation in Part 1 above).
    #
    # init.normal_(...) fills this region in-place with values drawn from
    # a normal distribution N(xm, xs²).
    #
    init.normal_(x[:, :, stx:stx + szx, sty:sty + szy], mean=xm, std=xs)

    # --- Clamp values to valid range ---
    #
    # x.clamp_(mn, mx)
    #   Clips ALL values in x to be between mn and mx.
    #   Any value below mn is set to mn; any value above mx is set to mx.
    #
    #   The underscore _ means in-place (modifies x directly).
    #
    #   WHY IS CLAMPING NEEDED?
    #   The random noise from init.normal_ can produce extreme values
    #   (e.g., very bright or very dark pixels that are outside the range
    #   of the original image data). Without clamping:
    #     - Values outside the training data range would be "alien" to the model
    #     - Could destabilize training (weird gradients from extreme values)
    #     - Could produce visual artifacts if you display the images
    #
    #   mn and mx are typically computed from the image batch:
    #     mn = x.min()  → the darkest pixel value in the batch
    #     mx = x.max()  → the brightest pixel value in the batch
    #   This ensures the noise stays within the natural range of the images.
    #
    #   Example:
    #     If image values range from -0.8 to 2.1 (after normalization),
    #     and noise generates a value of 3.5, it gets clamped to 2.1.
    #
    x.clamp_(mn, mx)


# ---- Test _rand_erase1 ----
#
# Get a fresh batch of images (the previous xbt was already modified)
xb, _ = next(iter(dls.train))
xbt = xb[:16]

# Call _rand_erase1 with statistics computed from the batch.
#
# Arguments:
#   xbt             — the 16 images to modify
#   0.2             — erase 20% of each dimension
#   xbt.mean()      — mean pixel value for the noise
#   xbt.std()       — std for the noise
#   xbt.min()       — minimum pixel value for clamping
#   xbt.max()       — maximum pixel value for clamping
#
# After this call, xbt is modified in-place with one random erased rectangle.
#
_rand_erase1(xbt, 0.2, xbt.mean(), xbt.std(), xbt.min(), xbt.max())

# Display the modified images — you'll see 16 images with a small noisy patch
show_images(xbt, imsize=1.5)


# =============================================================================
# PART 3: rand_erase — ERASE MULTIPLE RANDOM RECTANGLES
# =============================================================================
#
# One erased rectangle might not be enough to prevent the model from overfitting.
# By erasing MULTIPLE rectangles in random positions, we create more diverse
# training samples and make the augmentation more effective.
#
# The function rand_erase() erases a RANDOM number of rectangles (0 to max_num).
# The randomness in the COUNT is important:
#   - Sometimes 0 rectangles are erased (image unchanged) — the model still
#     needs to handle clean images correctly!
#   - Sometimes 1-4 rectangles are erased in different positions
# This variety prevents the model from learning "expect N erased rectangles."
#
# =============================================================================

#|export
def rand_erase(x, pct=0.2, max_num=4):
    """
    Randomly erase 0 to max_num rectangular regions from a batch of images.

    Each erased region is independently positioned and filled with
    Gaussian noise matching the image statistics. The number of regions
    erased is also random (uniformly distributed from 0 to max_num).

    Args:
        x:       Image tensor, shape (batch, channels, height, width).
        pct:     Float, fraction of image dimension for each rectangle.
                 Default 0.2 means each rectangle is ~20% of image height/width.
        max_num: Int, maximum number of rectangles to erase.
                 Default 4. The actual number is random from 0 to max_num.

    Returns:
        x — the modified image tensor (also modified in-place, but returning
            it allows chaining: result = rand_erase(images)
    """

    # --- Pre-compute image statistics ONCE ---
    #
    # We compute mean, std, min, max from the ENTIRE batch BEFORE erasing.
    # These statistics are passed to each _rand_erase1 call.
    #
    # WHY COMPUTE ONCE?
    # 1. Efficiency: Computing .mean(), .std(), .min(), .max() over the full
    #    tensor is expensive. Doing it once is faster than doing it N times.
    # 2. Consistency: If we computed stats AFTER each erase, the statistics
    #    would change (because we've modified the tensor). Using the original
    #    stats ensures all noise patches have consistent properties.
    #
    # xm = mean of all pixel values across the batch
    # xs = standard deviation of all pixel values
    # mn = minimum pixel value (for clamping noise to valid range)
    # mx = maximum pixel value (for clamping noise to valid range)
    #
    xm, xs, mn, mx = x.mean(), x.std(), x.min(), x.max()

    # --- Randomly choose HOW MANY rectangles to erase ---
    #
    # random.randint(0, max_num)
    #   Returns a random integer N where 0 ≤ N ≤ max_num (both inclusive).
    #   NOTE: Python's random.randint is INCLUSIVE on both ends.
    #   So random.randint(0, 4) can return: 0, 1, 2, 3, or 4.
    #
    #   When num=0: No rectangles are erased. The image passes through unchanged.
    #   This is intentional — it adds more variety and means the model can't
    #   assume that every training image is augmented.
    #
    num = random.randint(0, max_num)

    # --- Erase 'num' rectangles one at a time ---
    #
    # Each call to _rand_erase1 picks a NEW random position and fills it.
    # The rectangles may overlap (that's fine — it just means some pixels
    # get erased twice, which has no negative effect).
    #
    # IMPORTANT: Each _rand_erase1 erases the SAME rectangle across ALL
    # images in the batch (because it uses [:, :, ...] slicing). However,
    # different calls erase DIFFERENT rectangles (because random.random()
    # gives different positions each time).
    #
    for i in range(num):
        _rand_erase1(x, pct, xm, xs, mn, mx)

    # Return the modified tensor.
    # Note: x was already modified in-place by _rand_erase1, so returning it
    # is technically optional. But returning it allows functional-style usage:
    #   augmented = rand_erase(images)
    # This is a common PyTorch convention for in-place-but-also-returns functions.
    return x


# ---- Test rand_erase with multiple rectangles ----
#
# Get a fresh batch (previous one was already erased)
xb, _ = next(iter(dls.train))
xbt = xb[:16]

# Apply rand_erase: will erase between 0 and 4 random rectangles
# Each rectangle is ~20% of image size (5x5 pixels on 28x28 images)
rand_erase(xbt, 0.2, 4)

# Display — you'll see images with 0-4 small noisy patches scattered randomly
show_images(xbt, imsize=1.5)


# =============================================================================
# PART 4: RandErase — PYTORCH MODULE WRAPPER
# =============================================================================
#
# WHY WRAP IN A MODULE?
# ---------------------
# The rand_erase function works, but to use it in an augmentation PIPELINE
# with other transforms, we need it to be a PyTorch nn.Module.
#
# PyTorch's nn.Sequential is like a recipe that chains transforms together:
#   tfms = nn.Sequential(
#       transforms.RandomCrop(28, padding=1),      # Step 1: random crop
#       transforms.RandomHorizontalFlip(),           # Step 2: random flip
#       RandErase()                                  # Step 3: random erase
#   )
#
# nn.Sequential requires each step to be an nn.Module (it calls .forward()
# on each one in sequence). Wrapping rand_erase in a Module class lets it
# plug into this pipeline seamlessly.
#
# DESIGN PATTERN:
# This is a common pattern in PyTorch and fast.ai:
#   1. Write a plain function (rand_erase) — easy to test and understand
#   2. Wrap it in an nn.Module (RandErase) — easy to compose with other modules
#   3. Use it in nn.Sequential — clean, declarative augmentation pipelines
#
# =============================================================================

#|export
class RandErase(nn.Module):
    """
    Random Erase as a PyTorch Module.

    Wraps the rand_erase() function so it can be used inside nn.Sequential
    for composable augmentation pipelines.

    Usage:
        tfms = nn.Sequential(
            transforms.RandomCrop(28, padding=1),
            transforms.RandomHorizontalFlip(),
            RandErase(pct=0.2, max_num=4)
        )

    Args:
        pct:     Fraction of image dimension for each erased rectangle.
                 Default: 0.2 (20% of height/width).
        max_num: Maximum number of rectangles to erase per image.
                 Default: 4. Actual number is random from 0 to max_num.
    """

    def __init__(self, pct=0.2, max_num=4):
        # super().__init__() MUST be called in every nn.Module subclass.
        # It initializes the base Module class, which handles:
        #   - Parameter registration (for learnable params, though we have none here)
        #   - Sub-module tracking
        #   - Device management (CPU/GPU)
        #   - Training/eval mode switching
        #   - Hooks and other PyTorch internals
        #
        # Without this call, many PyTorch features would break silently.
        #
        super().__init__()

        # Store pct and max_num as instance attributes so forward() can access them.
        #
        # self.pct, self.max_num = pct, max_num
        #   This is Python's multiple assignment syntax. It's equivalent to:
        #     self.pct = pct
        #     self.max_num = max_num
        #   Just more concise.
        #
        self.pct, self.max_num = pct, max_num

    def forward(self, x):
        # forward() is the method that nn.Sequential calls when processing data.
        # When PyTorch encounters this module in a Sequential pipeline, it calls:
        #   output = module(input)
        # which internally calls module.forward(input).
        #
        # All we do is delegate to the rand_erase function with our stored parameters.
        #
        # x: input image tensor, shape (batch, channels, height, width)
        # Returns: modified image tensor (same shape, but with erased rectangles)
        #
        return rand_erase(x, self.pct, self.max_num)


# =============================================================================
# PART 5: FULL AUGMENTATION PIPELINE — COMBINING EVERYTHING
# =============================================================================
#
# Now we assemble a complete augmentation pipeline that chains three transforms
# together: RandomCrop → RandomHorizontalFlip → RandErase. This is the pipeline
# that would be used during actual training.
#
# =============================================================================

# ---- Create the augmentation pipeline ----
#
# nn.Sequential(transform1, transform2, transform3)
#   Creates a chain of transforms that are applied one after another.
#   When called with an image batch, it does:
#     result = transform3(transform2(transform1(input)))
#
# The three transforms:
#
#   1. transforms.RandomCrop(28, padding=1)
#      First PADS the image by 1 pixel on each side (28x28 → 30x30),
#      then randomly crops back to 28x28. This effectively shifts the image
#      by up to 1 pixel in any direction.
#
#      WHY?
#      The model should recognize a T-shirt whether it's centered perfectly
#      or shifted slightly. This teaches position invariance.
#
#      padding=1 is subtle (only 1 pixel shift). The earlier notebook section
#      used padding=4 for more aggressive shifting.
#
#   2. transforms.RandomHorizontalFlip()
#      50% chance to flip the image left-to-right. Uses the default
#      probability of 0.5 (not specified, so it uses the torchvision default).
#
#      WHY?
#      A T-shirt looks the same whether facing left or right. Flipping
#      doubles the effective dataset size for horizontally symmetric objects.
#      (Note: For digits or text, horizontal flip would be BAD because
#      "b" flipped becomes "d"! Always consider your data.)
#
#   3. RandErase()
#      Our custom module (defined above). Erases 0-4 random rectangles,
#      each covering ~20% of image dimensions. Uses default pct=0.2, max_num=4.
#
#      WHY?
#      Forces the model to learn from all parts of the image, not just
#      the most distinctive features. Reduces overfitting.
#
tfms = nn.Sequential(
    transforms.RandomCrop(28, padding=1),    # Slight random shift
    transforms.RandomHorizontalFlip(),        # 50% chance horizontal flip
    RandErase()                               # Random erase 0-4 rectangles
)

# ---- Create the augmentation callback ----
#
# augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)
#
# Let's unpack this step by step:
#
# partial(tfm_batch, tfm_x=tfms)
#   Creates a new function that is tfm_batch with tfm_x pre-set to our pipeline.
#
#   tfm_batch is defined as:
#     def tfm_batch(b, tfm_x=fc.noop, tfm_y=fc.noop):
#         return tfm_x(b[0]), tfm_y(b[1])
#   It applies separate transforms to images (b[0]) and labels (b[1]).
#
#   partial(tfm_batch, tfm_x=tfms) creates:
#     def new_func(b, tfm_y=fc.noop):
#         return tfms(b[0]), tfm_y(b[1])
#
#   So when this function receives a batch, it:
#     1. Applies RandomCrop → RandomHorizontalFlip → RandErase to the IMAGES
#     2. Leaves the LABELS unchanged (fc.noop = identity = "do nothing")
#
#   WHY USE partial()?
#   BatchTransformCB expects a function that takes a batch (images, labels)
#   and returns a transformed batch. partial() adapts tfm_batch to match
#   this interface while baking in our specific augmentation pipeline.
#
# BatchTransformCB(transform_func, on_val=False)
#   A callback that applies transform_func to every batch during training.
#
#   on_val=False is CRITICAL:
#     This means augmentation is ONLY applied during TRAINING, not validation.
#
#     WHY?
#     During validation, we want to evaluate the model on CLEAN, unmodified
#     images to get an honest measure of performance. If we augmented
#     validation images too, we'd be measuring how well the model handles
#     corrupted images, which isn't what we want.
#
#     During training, augmentation artificially increases dataset diversity,
#     helping the model generalize. But during validation/testing, we want
#     to measure real-world performance on actual images.
#
#     (Exception: TTA deliberately sets on_val=True — see the TTA section)
#
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)


# ---- Visualize the augmented batch ----
#
# We create a quick test setup to see what the augmented images look like.
#

# get_model() creates a fresh ResNet model (defined earlier in the notebook).
# We don't care about the model's predictions here — we just need a Learner
# to run the data through the augmentation pipeline for visualization.
model = get_model()

# Create a TrainLearner with specific callbacks for visualization:
#
# TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=[...])
#   model:           the ResNet model
#   dls:             DataLoaders with train and validation sets
#   F.cross_entropy: loss function (needed by Learner, but we won't actually train)
#   lr=lr:           learning rate (needed by Learner, but irrelevant here)
#   cbs=[...]:       list of callbacks — this is where the magic happens
#
# The callbacks:
#
#   DeviceCB()
#     Moves each batch to the GPU (or CPU if no GPU). Required because the model
#     is on a device and the data needs to be on the same device.
#
#   SingleBatchCB()
#     STOPS training after just ONE batch. This is a debugging/visualization
#     callback — instead of running through the entire dataset, it processes
#     one batch and then raises an exception to halt .fit(). The Learner
#     catches this exception and stores the last batch in learn.batch.
#     This is perfect for quickly testing augmentations without waiting
#     for a full epoch.
#
#   augcb
#     Our augmentation callback defined above. It applies RandomCrop +
#     RandomHorizontalFlip + RandErase to the training batch.
#
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr,
                     cbs=[DeviceCB(), SingleBatchCB(), augcb])

# learn.fit(1)
#   Starts the training loop. Because SingleBatchCB is active, it processes
#   exactly one batch and then stops. After this call:
#     learn.batch = (augmented_images, labels)
#   The images have been through the full augmentation pipeline.
learn.fit(1)

# Extract the augmented images and labels from the captured batch.
#
# learn.batch is a tuple: (images_tensor, labels_tensor)
#   xb shape: (1024, 1, 28, 28) — full batch of augmented images
#   yb shape: (1024,) — corresponding labels (unchanged by augmentation)
#
xb, yb = learn.batch

# Display the first 16 augmented images in a grid.
#
# xb[:16] — slice the first 16 images
# imsize=1.5 — each image is 1.5 inches in the display
#
# YOU SHOULD SEE:
# 16 Fashion MNIST images that have been:
#   1. Slightly shifted (RandomCrop with 1 pixel padding)
#   2. Some are horizontally flipped (RandomHorizontalFlip)
#   3. Some have 1-4 small noisy/grainy rectangles (RandErase)
# The combination of all three makes each image a unique variation
# of the original, effectively expanding the training dataset.
#
show_images(xb[:16], imsize=1.5)


# =============================================================================
# VISUAL SUMMARY: THE COMPLETE RANDOM ERASE PIPELINE
# =============================================================================
#
#   Original Image (28x28)
#          │
#          ▼
#   ┌─────────────────┐
#   │  RandomCrop      │   Pad 1px each side → 30x30
#   │  (28, padding=1) │   Randomly crop back to 28x28
#   └────────┬────────┘   Effect: image shifts up to 1px
#            │
#            ▼
#   ┌─────────────────┐
#   │  RandomHFlip     │   50% chance to flip left↔right
#   │  (p=0.5)         │
#   └────────┬────────┘
#            │
#            ▼
#   ┌─────────────────┐
#   │  RandErase       │   Choose random N in [0, 4]
#   │  (pct=0.2,       │   For each of N rectangles:
#   │   max_num=4)     │     - Random position
#   └────────┬────────┘     - Fill with Gaussian noise
#            │               - Clamp to valid range
#            ▼
#   Augmented Image (28x28)
#   → Fed to model for training
#
# =============================================================================
#
# KEY DESIGN DECISIONS:
#
# 1. NOISE vs ZEROS:
#    We fill with Gaussian noise (matching image stats) rather than zeros.
#    Zeros would create obvious black rectangles that the model could learn
#    to detect and ignore, defeating the purpose of augmentation.
#
# 2. SAME REGION ACROSS BATCH:
#    Each call to _rand_erase1 erases the SAME rectangle position in all
#    images of the batch. This is a deliberate simplification that's fast
#    (one random number per call, not per image) and works well in practice.
#    Per-image random positions would be more diverse but slower.
#
# 3. RANDOM NUMBER OF RECTANGLES:
#    Choosing 0-4 rectangles randomly means:
#    - ~20% of batches get 0 rectangles (clean pass-through)
#    - ~20% get 1, ~20% get 2, ~20% get 3, ~20% get 4
#    This variety prevents the model from learning to expect a fixed pattern.
#
# 4. AUGMENTATION ONLY DURING TRAINING (on_val=False):
#    Validation uses clean images for honest performance measurement.
#    Training uses augmented images for better generalization.
#
# =============================================================================

# Random Erasing: Teaching the Model Not to Rely on Any Single Region

## The Problem: Models Can Be Lazy

Neural networks are surprisingly good at finding shortcuts. If a shoe always has a distinctive sole visible in the bottom-left of the image, the model might learn to just look at that one spot and ignore everything else. This is a problem because:

- In the real world, that region might be occluded (blocked by another object)
- The image might be cropped differently
- The model becomes fragile — change one small region and it falls apart

**What if we forced the model to NOT rely on any single region of the image?**

That's the idea behind **Random Erasing** (also called "Cutout"). During training, we randomly blank out a rectangular patch of each image with noise. The model can't predict what region will be erased next, so it's forced to learn **redundant representations** — it must recognize a T-shirt from the collar alone, from the sleeves alone, from the pattern alone, from any combination. This makes the model significantly more robust.

## Step-by-Step: Building Random Erasing from Scratch

### Getting Some Images to Work With

```python
# Get a batch of images
xb, _ = next(iter(dls.train))
xbt = xb[:16]
```

**`next(iter(dls.train))`** — grabs one batch from the training dataloader. `dls.train` is our training DataLoader, `iter(...)` creates an iterator over it, and `next(...)` pulls out the first batch. A batch is a tuple of `(images, labels)`.

**`xb, _ = ...`** — we unpack the tuple. `xb` gets the images, `_` is a Python convention meaning "I don't care about this value" (we're ignoring the labels because we just want to experiment with the images).

**`xbt = xb[:16]`** — takes just the first 16 images from the batch (out of 1024). We use a small subset so we can easily visualize what's happening. `xbt` shape: `(16, 1, 28, 28)` — 16 grayscale 28×28 images.

### Computing Image Statistics

```python
# Get stats for replacing erased region
xm, xs = xbt.mean(), xbt.std()
```

When we erase a region, we need to fill it with *something*. We could use zeros (black pixels), but that would create an obvious unnatural patch that the model could learn to recognize and ignore. Instead, we fill with **random noise** that has the **same mean and standard deviation** as the real image data.

- `xm = xbt.mean()` — the average pixel value across all 16 images. Since our images are normalized (centered around 0), this will be close to 0.
- `xs = xbt.std()` — the spread of pixel values. This tells us how "wide" the distribution is.

By matching the mean and std, the noise patch blends in statistically with the rest of the image. The model can't easily distinguish "real pixels" from "noise pixels" based on their overall statistics.

### Choosing What Percentage to Erase

```python
# Percentage of image to erase
pct = 0.2  # 20%
```

We'll erase a rectangle that covers about 20% of the image's height and 20% of its width. For a 28×28 image, that's roughly a 5×5 pixel patch. This is a good balance — large enough to actually hide meaningful features, small enough that most of the image remains visible so the model can still learn.

### Calculating the Random Position and Size

```python
# Calculate random position and size for erasing
szx = int(pct * xbt.shape[-2])  # Height of erased region
szy = int(pct * xbt.shape[-1])  # Width of erased region
stx = int(random.random() * (1-pct) * xbt.shape[-2])  # Start row
sty = int(random.random() * (1-pct) * xbt.shape[-1])  # Start column
```

Let me walk through each line carefully.

**The image dimensions:**
Our tensor `xbt` has shape `(16, 1, 28, 28)` meaning `(batch, channels, height, width)`.
- `xbt.shape[-2]` = 28 — the height (second-to-last dimension)
- `xbt.shape[-1]` = 28 — the width (last dimension)

**Calculating the size of the rectangle to erase:**
- `szx = int(0.2 * 28)` = `int(5.6)` = **5** — the erased region is 5 pixels tall
- `szy = int(0.2 * 28)` = `int(5.6)` = **5** — the erased region is 5 pixels wide

**Calculating a random starting position:**

We need the rectangle to fit entirely within the image. If the image is 28 pixels wide and our rectangle is 5 pixels wide, the starting column can be anywhere from 0 to 23 (so that 23+5=28 doesn't go out of bounds).

- `random.random()` — generates a random float between 0.0 and 1.0
- `(1-pct)` = 0.8 — this is the fraction of the image where the rectangle can start while still fitting. If the rectangle takes 20% of the width, the start position can be anywhere in the first 80%.
- `stx = int(random.random() * 0.8 * 28)` — a random row between 0 and 22
- `sty = int(random.random() * 0.8 * 28)` — a random column between 0 and 22

Every time this code runs, `stx` and `sty` will be different random values, so the erased patch lands in a different spot each time.

### Actually Erasing (Filling with Noise)

```python
# Fill erased region with random noise (same mean/std as image)
init.normal_(xbt[:, :, stx:stx+szx, sty:sty+szy], mean=xm, std=xs);
```

This is the core operation. Let me break down the indexing:

**`xbt[:, :, stx:stx+szx, sty:sty+szy]`** — this selects the rectangular region to erase:
- `:`  — all 16 images in the batch (the SAME region in every image)
- `:`  — all channels (just 1 for grayscale, would be 3 for RGB)
- `stx:stx+szx` — rows from `stx` to `stx+5` (the height of our rectangle)
- `sty:sty+szy` — columns from `sty` to `sty+5` (the width of our rectangle)

**`init.normal_(..., mean=xm, std=xs)`** — fills this selected region **in-place** (the trailing `_` means "modify the tensor directly, don't create a copy") with random values drawn from a normal (Gaussian) distribution with the same mean and standard deviation as our image data. `init` here is `torch.nn.init`, a module that provides various tensor initialization strategies.

The semicolon `;` at the end just suppresses the output in Jupyter (so the notebook doesn't print the tensor).

### Visualizing the Result

```python
show_images(xbt, imsize=1.5)
```

This displays the 16 images with their erased patches. We should see each image with a small noisy rectangle somewhere — the same position in all 16 images (since `stx` and `sty` were computed once).

## Packaging It Up: The `_rand_erase1` Function

Now we turn the manual steps above into a reusable function:

```python
#|export
def _rand_erase1(x, pct, xm, xs, mn, mx):
    szx = int(pct * x.shape[-2])
    szy = int(pct * x.shape[-1])
    stx = int(random.random() * (1-pct) * x.shape[-2])
    sty = int(random.random() * (1-pct) * x.shape[-1])
    init.normal_(x[:, :, stx:stx+szx, sty:sty+szy], mean=xm, std=xs)
    x.clamp_(mn, mx)
```

This does exactly what we did manually, with one important addition:

**`x.clamp_(mn, mx)`** — after filling with random noise, some generated values might be outside the valid range for our images. `clamp_` forces every pixel value to be between `mn` (the minimum value in the original data) and `mx` (the maximum value). This prevents the noise from creating extreme outlier values that could confuse the model or cause numerical issues.

The function name starts with `_` (underscore), which is a Python convention meaning "this is a private/internal helper function, not meant to be called directly by users."

**Arguments:**
- `x` — the batch of image tensors to modify
- `pct` — what fraction of the image dimensions to erase (0.2 = 20%)
- `xm, xs` — mean and std for generating replacement noise
- `mn, mx` — min and max values to clamp the result to

### Testing the single-erase function

```python
xb, _ = next(iter(dls.train))
xbt = xb[:16]
_rand_erase1(xbt, 0.2, xbt.mean(), xbt.std(), xbt.min(), xbt.max())
show_images(xbt, imsize=1.5)
```

We grab a fresh batch, take 16 images, erase one random rectangle, and visualize. Each time we run this cell, the rectangle will appear in a different random position.

## Going Further: Multiple Random Erases

One erased rectangle is good, but we can do better. By erasing **multiple** random rectangles, we force the model to be robust against losing information in **several** places at once.

```python
#|export
def rand_erase(x, pct=0.2, max_num=4):
    xm, xs, mn, mx = x.mean(), x.std(), x.min(), x.max()
    num = random.randint(0, max_num)
    for i in range(num):
        _rand_erase1(x, pct, xm, xs, mn, mx)
    return x
```

Let me walk through the logic:

**`xm, xs, mn, mx = x.mean(), x.std(), x.min(), x.max()`** — compute the image statistics once, before any erasing. We calculate these upfront (not inside the loop) because we want the noise to match the **original** image statistics, not the statistics after some regions have already been erased.

**`num = random.randint(0, max_num)`** — randomly choose how many rectangles to erase, anywhere from **0 to 4** (inclusive). This randomness is important:
- Sometimes we erase 0 patches (the image is untouched) — so the model still sees clean images
- Sometimes we erase 4 patches — the model must cope with significant occlusion
- This variability prevents the model from "expecting" a specific number of patches

**`for i in range(num): _rand_erase1(x, pct, xm, xs, mn, mx)`** — erases `num` rectangles one at a time. Each call to `_rand_erase1` picks a new random position, so the rectangles end up in different places (and might even overlap, which is fine).

**`return x`** — returns the modified images. Note that `x` was already modified in-place by `_rand_erase1`, but returning it explicitly makes the function work cleanly in pipelines.

### Testing multiple erases

```python
xb, _ = next(iter(dls.train))
xbt = xb[:16]
rand_erase(xbt, 0.2, 4)
show_images(xbt, imsize=1.5)
```

Now we should see images with 0 to 4 noisy rectangles scattered randomly across them.

## Making It a PyTorch Module: `RandErase`

To use random erasing as part of an augmentation pipeline (alongside crops, flips, etc.), we wrap it in a `nn.Module`:

```python
#|export
class RandErase(nn.Module):
    def __init__(self, pct=0.2, max_num=4):
        super().__init__()
        self.pct, self.max_num = pct, max_num

    def forward(self, x):
        return rand_erase(x, self.pct, self.max_num)
```

**Why wrap it in `nn.Module`?**

PyTorch's `nn.Sequential` only works with `nn.Module` subclasses. By wrapping our function in a module, we can chain it with other transforms like `RandomCrop` and `RandomHorizontalFlip` in a single pipeline. 

**`__init__`** — stores the configuration (what percentage to erase, max number of patches). `super().__init__()` is required boilerplate that initializes the parent `nn.Module` class.

**`forward`** — the method PyTorch calls when data flows through this module. It simply delegates to our `rand_erase` function.

## Putting It All Together: The Augmentation Pipeline

```python
tfms = nn.Sequential(
    transforms.RandomCrop(28, padding=1),
    transforms.RandomHorizontalFlip(),
    RandErase()
)
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)
```

Here we build a complete data augmentation pipeline that applies **three** different augmentations in sequence:

**`transforms.RandomCrop(28, padding=1)`** — First, pad the image by 1 pixel on each side (making it 30×30 with the padded border), then randomly crop it back to 28×28. This effectively shifts the image by 0–1 pixels in any direction. Why? It teaches the model that an object doesn't have to be perfectly centered — a shoe shifted 1 pixel to the left is still a shoe.

**`transforms.RandomHorizontalFlip()`** — Randomly flips the image horizontally (left ↔ right) with 50% probability. A pullover facing left is the same class as one facing right.

**`RandErase()`** — Our random erasing with default settings (erase 20% patches, up to 4 of them). Forces the model to be robust to missing regions.

**`nn.Sequential(...)`** wraps all three into a single callable pipeline. When data flows through, it hits each transform in order: crop → flip → erase.

**`augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)`** — creates a callback that applies our pipeline to every training batch:
- `partial(tfm_batch, tfm_x=tfms)` — creates a function that applies `tfms` to the input images (`tfm_x`) within each batch
- **`on_val=False`** — this is crucial! Augmentation should ONLY happen during **training**, NOT during validation/testing. During validation, we want to evaluate on clean, unmodified images to get an honest measure of how good the model is. If we augmented validation images too, we'd be measuring performance on corrupted data, which isn't meaningful.

### Visualizing the Full Pipeline

```python
model = get_model()
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=[DeviceCB(), SingleBatchCB(), augcb])
learn.fit(1)
xb, yb = learn.batch
show_images(xb[:16], imsize=1.5)
```

**`get_model()`** — creates a fresh model (we're not interested in training here, just seeing the augmentations).

**`SingleBatchCB()`** — a callback that stops after just one batch. We only need one batch to see what the augmented images look like.

**`augcb`** — our augmentation callback from above, injected into the training loop.

**`learn.fit(1)`** — runs one "epoch" (but `SingleBatchCB` stops it after one batch). During this single batch, the augmentation pipeline fires: each image gets randomly cropped, maybe flipped, and then has 0–4 random rectangles erased.

**`xb, yb = learn.batch`** — grabs the batch that was just processed (after augmentation has been applied). `xb` is the augmented images, `yb` is the labels.

**`show_images(xb[:16], imsize=1.5)`** — displays the first 16 augmented images. We should see images that are slightly shifted, some horizontally flipped, and with noisy rectangles in random positions. These are what the model actually trains on — slightly different every epoch, forcing it to learn robust features.

## Why This Combination of Augmentations Works So Well

Each augmentation teaches the model a different kind of robustness:

```
┌─────────────────────┬──────────────────────────────────────────────┐
│ Augmentation        │ What It Teaches the Model                   │
├─────────────────────┼──────────────────────────────────────────────┤
│ RandomCrop          │ "Objects aren't always perfectly centered"   │
│ RandomHorizontalFlip│ "Objects can face either direction"          │
│ RandErase           │ "You must recognize objects from partial     │
│                     │  information — don't rely on one region"     │
└─────────────────────┴──────────────────────────────────────────────┘
```

Together, they dramatically reduce overfitting and improve generalization. The model sees a different variation of each image every epoch, effectively multiplying the size of our training set. An image that appears 5 times across 5 epochs will look slightly different each time — different crop, maybe flipped, different patches erased. The model is forced to learn the **essence** of each class, not memorize specific pixel arrangements.

---
## Part 8: Random Copy

**Random Copy** copies a patch from one location to another within the same image. This is an alternative to Random Erase.

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. It loads ./interactive_viz/random_copy_lab.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Random Copy (_rand_copy1): CLICK once for the source patch, again for the
# destination — and see why copying preserves image statistics EXACTLY.
# ============================================================================
show_viz("interactive_viz/random_copy_lab.html", height="760px")

In [ ]:
#|export
def _rand_copy1(x, pct):
    """
    Copy a random patch from one location to another.
    
    Args:
        x: Image tensor
        pct: Percentage of image for patch size
    """
    # Patch size
    szx = int(pct * x.shape[-2])
    szy = int(pct * x.shape[-1])
    
    # Source position (where to copy from)
    stx1 = int(random.random() * (1-pct) * x.shape[-2])
    sty1 = int(random.random() * (1-pct) * x.shape[-1])
    
    # Destination position (where to copy to)
    stx2 = int(random.random() * (1-pct) * x.shape[-2])
    sty2 = int(random.random() * (1-pct) * x.shape[-1])
    
    # Copy patch
    x[:, :, stx1:stx1+szx, sty1:sty1+szy] = x[:, :, stx2:stx2+szx, sty2:sty2+szy]

In [ ]:
# Test
xb, _ = next(iter(dls.train))
xbt = xb[:16]
_rand_copy1(xbt, 0.2)
show_images(xbt, imsize=1.5)

In [ ]:
#|export
def rand_copy(x, pct=0.2, max_num=4):
    """
    Randomly copy 0 to max_num patches within the image.
    
    Args:
        x: Image tensor
        pct: Percentage of image for each patch
        max_num: Maximum number of patches to copy
    
    Returns:
        Modified image tensor
    """
    num = random.randint(0, max_num)
    for i in range(num): 
        _rand_copy1(x, pct)
    return x

In [ ]:
#|export
class RandCopy(nn.Module):
    """
    Random Copy as a PyTorch module.
    
    Args:
        pct: Percentage of image for each patch (default: 0.2)
        max_num: Maximum number of patches to copy (default: 4)
    """
    def __init__(self, pct=0.2, max_num=4):
        super().__init__()
        self.pct, self.max_num = pct, max_num
    
    def forward(self, x): 
        return rand_copy(x, self.pct, self.max_num)

In [ ]:
# Create augmentation with RandCopy
tfms = nn.Sequential(
    transforms.RandomCrop(28, padding=1),
    transforms.RandomHorizontalFlip(),
    RandCopy()
)
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)

In [ ]:
# Visualize
model = get_model()
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=[DeviceCB(), SingleBatchCB(), augcb])
learn.fit(1)
xb, yb = learn.batch
show_images(xb[:16], imsize=1.5)

---
## Part 9: Dropout

**Dropout** randomly zeros out activations during training. This prevents co-adaptation of neurons and acts as regularization.

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. It loads ./interactive_viz/dropout_lab.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Dropout: sample the Bernoulli mask → multiply → scale by 1/(1-p). Click
# neurons to drop them, flip train/eval mode, and watch E[output] stay equal to E[x].
# ============================================================================
show_viz("interactive_viz/dropout_lab.html", height="740px")

In [ ]:
# Dropout probability
p = 0.1  # 10% of activations will be dropped

# Binomial distribution: each element has (1-p) probability of being kept
dist = distributions.binomial.Binomial(probs=1-p)
dist.sample((10,))  # Sample 10 values (1 = keep, 0 = drop)

tensor([1., 0., 1., 1., 1., 1., 1., 1., 1., 1.])

In [ ]:
class Dropout(nn.Module):
    """
    Dropout layer implementation.
    
    During training:
    - Randomly zero out p fraction of activations
    - Scale remaining activations by 1/(1-p) to maintain expected value
    
    During inference:
    - Pass through unchanged (all activations used)
    
    Args:
        p: Dropout probability (default: 0.1)
    
    The scaling by 1/(1-p) is called "inverted dropout" and ensures
    that expected values are the same during training and inference.
    """
    def __init__(self, p=0.1):
        super().__init__()
        self.p = p

    def forward(self, x):
        # During inference, don't drop anything
        if not self.training: 
            return x
        
        # Create binomial mask (1 = keep, 0 = drop)
        # probs=1-p means each element has (1-p) probability of being 1
        dist = distributions.binomial.Binomial(
            tensor(1.0).to(x.device),  # Single trial
            probs=1 - self.p           # Keep probability
        )
        
        # Apply mask and scale
        # Multiply by 1/(1-p) so expected value remains the same
        return x * dist.sample(x.size()) * 1/(1 - self.p)

### Why Scale by 1/(1-p)?

```
Without scaling:
    Training:  E[output] = x * (1-p)  (p fraction is zeroed)
    Inference: E[output] = x          (nothing zeroed)
    Problem: Different expected values!

With scaling by 1/(1-p):
    Training:  E[output] = x * (1-p) * 1/(1-p) = x
    Inference: E[output] = x
    Both have same expected value!
```

In [ ]:
def get_dropmodel(act=nn.ReLU, nfs=(16, 32, 64, 128, 256, 512), norm=nn.BatchNorm2d, drop=0.0):
    """
    Model with dropout layers.
    
    Adds dropout:
    - Dropout2d after first ResBlock (drops entire channels)
    - Regular Dropout before final linear layer
    
    Args:
        act: Activation function class
        nfs: Filter counts per layer
        norm: Normalization layer class
        drop: Dropout probability (default: 0 = no dropout)
    """
    layers = [
        ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm), 
        nn.Dropout2d(drop)  # Drops entire channels
    ]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) 
               for i in range(len(nfs)-1)]
    layers += [
        nn.Flatten(), 
        Dropout(drop),  # Regular dropout
        nn.Linear(nfs[-1], 10, bias=False), 
        nn.BatchNorm1d(10)
    ]
    return nn.Sequential(*layers)

In [ ]:
# iw = partial(init_weights, leaky=0.1)
"""

def init_weights(m, leaky=0.0):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        init.kaiming_normal_(m.weight, a=leaky)
    if isinstance(m, nn.BatchNorm2d):
        # BatchNorm gets its own initialization
        ...

"""

In [ ]:
# Train model with dropout
set_seed(42)
epochs = 5
lr = 1e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]

model = get_dropmodel(act_gr, norm=nn.BatchNorm2d, drop=0.1).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)

In [ ]:
learn.fit(epochs)

# Understanding `partial(init_weights)` and `.apply(iw)` — Explained from Scratch

## The Two Lines We're Explaining

```python
iw = partial(init_weights, leaky=0.1)
```

```python
model = get_dropmodel(act_gr, norm=nn.BatchNorm2d, drop=0.1).apply(iw)
```

These two lines do something fundamental: they **create a model** and then **initialize all its weights** using a specific strategy (Kaiming initialization). Let's break down every piece.

---

## Part 1: `iw = partial(init_weights, leaky=0.1)`

### What is `init_weights`?

`init_weights` is a function (defined in `miniai/init.py` from an earlier notebook) that initializes the weights of a **single layer** in a neural network. It looks roughly like this:

```python
def init_weights(m, leaky=0.0):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        init.kaiming_normal_(m.weight, a=leaky)
    if isinstance(m, nn.BatchNorm2d):
        # BatchNorm gets its own initialization
        ...
```

The function takes two arguments:

- **`m`** — a single PyTorch module (layer), like a `nn.Conv2d` or `nn.Linear`. The function checks what type of layer `m` is, and initializes its weights accordingly.
- **`leaky`** — the slope of the negative part of the activation function (for LeakyReLU / GeneralReLU). This matters because Kaiming initialization needs to know what activation function follows the layer in order to set the right scale for the weights. A standard ReLU has `leaky=0.0` (negative values are completely zeroed). Our `GeneralRelu` uses `leak=0.1` (negative values are scaled by 0.1), so we pass `leaky=0.1`.

### Why Kaiming Initialization Matters

When a neural network is first created, all its weights are random. But the **scale** of those random values matters enormously:

- **Too large** → activations explode (numbers grow to infinity), training diverges
- **Too small** → activations vanish (numbers shrink to zero), nothing is learned
- **Just right** → activations stay in a healthy range, training works smoothly

Kaiming initialization (also called "He initialization," after Kaiming He who proposed it) computes the mathematically correct scale based on the layer's fan-in (number of input connections) and the activation function. It ensures that the **variance of activations stays approximately constant** as data flows through the network, layer after layer.

### What is `partial`?

`partial` (from Python's `functools` module) creates a **new function** by "pre-filling" some arguments of an existing function. It's like placing a standing order at a coffee shop — instead of specifying your order every time, you pre-configure it once and then just say "the usual."

```python
from functools import partial

iw = partial(init_weights, leaky=0.1)
```

This creates a **new function** `iw` that is exactly like `init_weights`, except the `leaky` argument is already set to `0.1`. Here's what that means concretely:

```python
# BEFORE partial: init_weights needs TWO arguments
init_weights(some_layer, leaky=0.1)    # must specify leaky every time

# AFTER partial: iw only needs ONE argument (leaky is pre-filled)
iw = partial(init_weights, leaky=0.1)
iw(some_layer)                          # leaky=0.1 is automatic

# These two calls are IDENTICAL:
init_weights(some_layer, leaky=0.1)
iw(some_layer)
```

### Why Use `partial` Here?

The reason is that `.apply()` (which we'll use next) requires a function that takes **exactly one argument** — a single module. But `init_weights` takes **two arguments** (`m` and `leaky`). We can't pass `leaky=0.1` through `.apply()` — it doesn't support extra arguments.

`partial` solves this perfectly: it "bakes in" `leaky=0.1`, producing a new function `iw` that takes just one argument (`m`). Now `iw` is compatible with `.apply()`.

```
init_weights(m, leaky)   →  TWO arguments  →  ✗ incompatible with .apply()
                  │
          partial(init_weights, leaky=0.1)
                  │
                  ▼
iw(m)                    →  ONE argument   →  ✓ compatible with .apply()
```

---

## Part 2: `model = get_dropmodel(act_gr, norm=nn.BatchNorm2d, drop=0.1).apply(iw)`

This line does **two things** chained together:

1. `get_dropmodel(act_gr, norm=nn.BatchNorm2d, drop=0.1)` — creates the model
2. `.apply(iw)` — initializes all the model's weights

Let's take each in turn.

### Step 1: `get_dropmodel(...)` — Building the Model

```python
def get_dropmodel(act=nn.ReLU, nfs=(16, 32, 64, 128, 256, 512), norm=nn.BatchNorm2d, drop=0.0):
    layers = [
        ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm), 
        nn.Dropout2d(drop)
    ]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) 
               for i in range(len(nfs)-1)]
    layers += [
        nn.Flatten(), 
        Dropout(drop),
        nn.Linear(nfs[-1], 10, bias=False), 
        nn.BatchNorm1d(10)
    ]
    return nn.Sequential(*layers)
```

When we call `get_dropmodel(act_gr, norm=nn.BatchNorm2d, drop=0.1)`, here's what happens:

**The arguments:**

- `act_gr` = `partial(GeneralRelu, leak=0.1, sub=0.4)` — the activation function class to use. This is itself a `partial` — instead of using plain `nn.ReLU`, we use `GeneralReLU` with a leak of 0.1 and a subtraction of 0.4, pre-configured via `partial`.
- `norm=nn.BatchNorm2d` — use batch normalization
- `drop=0.1` — dropout probability of 10%

**The architecture that gets built (layer by layer):**

```
Layer 0:  ResBlock(1→16, 5×5 kernel, stride=1)    # First block, no downsampling
Layer 1:  Dropout2d(0.1)                            # Drop entire channels (10%)
Layer 2:  ResBlock(16→32, stride=2)                 # Downsample: 28×28 → 14×14
Layer 3:  ResBlock(32→64, stride=2)                 # 14×14 → 7×7
Layer 4:  ResBlock(64→128, stride=2)                # 7×7 → 4×4
Layer 5:  ResBlock(128→256, stride=2)               # 4×4 → 2×2
Layer 6:  ResBlock(256→512, stride=2)               # 2×2 → 1×1
Layer 7:  Flatten()                                  # (batch, 512, 1, 1) → (batch, 512)
Layer 8:  Dropout(0.1)                               # Drop individual neurons (10%)
Layer 9:  Linear(512→10)                             # 512 features → 10 classes
Layer 10: BatchNorm1d(10)                            # Normalize the 10 outputs
```

Each `ResBlock` itself contains multiple sub-layers (convolutions, batch norms, activations, skip connections). So the full model is a **tree** of modules — `nn.Sequential` at the top, `ResBlock`s inside, and `Conv2d`s and `BatchNorm2d`s inside those.

At this point, the model exists but all its weights are whatever PyTorch's default initialization gave them. Now we need to set them properly.

### Step 2: `.apply(iw)` — Initializing Every Layer's Weights

`.apply()` is a built-in PyTorch method available on every `nn.Module`. It does something beautifully simple: **it walks through every single sub-module in the entire model tree and calls the given function on each one.**

Here's a simplified version of how `.apply()` works internally:

```python
# Simplified version of nn.Module.apply()
def apply(self, fn):
    for child in self.children():    # iterate over direct child modules
        child.apply(fn)              # recursively apply to each child's children
    fn(self)                         # then apply to self
    return self                      # return self for method chaining
```

The key insight: `.apply()` is **recursive**. It doesn't just call `iw` on the top-level `nn.Sequential` — it drills down into every nested module. Our model is a tree:

```
nn.Sequential (the whole model)
├── ResBlock
│   ├── Conv2d          ← iw() is called on this
│   ├── BatchNorm2d     ← iw() is called on this
│   ├── GeneralReLU
│   ├── Conv2d          ← iw() is called on this
│   ├── BatchNorm2d     ← iw() is called on this
│   └── ...
├── Dropout2d
├── ResBlock
│   ├── Conv2d          ← iw() is called on this
│   ├── BatchNorm2d     ← iw() is called on this
│   └── ...
├── ... (more ResBlocks)
├── Flatten
├── Dropout
├── Linear              ← iw() is called on this
└── BatchNorm1d         ← iw() is called on this
```

`.apply(iw)` visits **every single node** in this tree and calls `iw(node)` on it. That means `iw` gets called on every `Conv2d`, every `BatchNorm2d`, every `Linear`, every `Dropout`, every `Flatten`, even the `ResBlock`s and `Sequential` themselves.

### What Happens Inside Each `iw()` Call

Remember, `iw` is `partial(init_weights, leaky=0.1)`, so calling `iw(m)` is the same as calling `init_weights(m, leaky=0.1)`. Inside `init_weights`, it checks the **type** of module and acts accordingly:

```python
def init_weights(m, leaky=0.0):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        init.kaiming_normal_(m.weight, a=leaky)
    # ... handle BatchNorm, etc.
```

So when `.apply(iw)` walks through the model:

| Module visited | What `iw` does |
|---|---|
| `Conv2d` | Reinitializes its weight tensor with Kaiming normal (using `a=0.1` for our leaky activation) |
| `Linear` | Same — Kaiming initialization on its weight tensor |
| `BatchNorm2d` | Sets weights to 1.0 and biases to 0.0 (standard BN init) |
| `BatchNorm1d` | Same as above |
| `Dropout`, `Dropout2d` | `isinstance` check fails → `iw` does **nothing** (dropout has no weights) |
| `Flatten` | `isinstance` check fails → `iw` does **nothing** (no weights) |
| `GeneralReLU` | `isinstance` check fails → `iw` does **nothing** (no weights) |
| `ResBlock` | `isinstance` check fails → `iw` does **nothing** (its children already handled) |
| `Sequential` | `isinstance` check fails → `iw` does **nothing** (its children already handled) |

This is the elegance of the `isinstance` check inside `init_weights` — it **silently skips** modules that don't have learnable weights. The function is safe to call on *any* module, and it only does something when it encounters the specific types it knows how to initialize.

### Why `.apply()` Returns `self`

Notice that `.apply(iw)` returns the model itself. This is what allows method chaining — we can write:

```python
model = get_dropmodel(act_gr, norm=nn.BatchNorm2d, drop=0.1).apply(iw)
```

as a single line instead of:

```python
model = get_dropmodel(act_gr, norm=nn.BatchNorm2d, drop=0.1)
model.apply(iw)   # modifies model in-place, returns model
```

Both are equivalent. The chained version is just more concise.

---

## Putting It All Together

Let's read the full line one more time with complete understanding:

```python
model = get_dropmodel(act_gr, norm=nn.BatchNorm2d, drop=0.1).apply(iw)
```

**In plain English:** "Create a ResNet model with dropout (10%), using GeneralReLU activations and BatchNorm. Then walk through every single layer in the model and reinitialize its weights using Kaiming initialization tuned for a leaky slope of 0.1. Store the result in `model`."

Here's the complete flow:

```
Step 1: partial(init_weights, leaky=0.1) → iw
        Pre-fill leaky=0.1 so iw takes only one argument

Step 2: get_dropmodel(act_gr, norm=nn.BatchNorm2d, drop=0.1)
        Build the model architecture (ResBlocks + Dropout + Linear + BN)
        Weights are PyTorch defaults at this point

Step 3: .apply(iw)
        Walk every sub-module recursively:
          Conv2d     → Kaiming init (a=0.1)
          Linear     → Kaiming init (a=0.1)
          BatchNorm  → weights=1, bias=0
          Everything else → skip (no weights to init)
        
        Weights are now properly initialized!

Step 4: model = ...
        The fully initialized model is stored in `model`,
        ready for training
```

### Why Not Just Use PyTorch's Default Initialization?

PyTorch does initialize weights automatically when you create a layer. For example, `nn.Conv2d` uses Kaiming uniform initialization by default. So why do we bother with this manual step?

The reason is **customization**. The default initialization assumes a standard ReLU activation (`leaky=0.0`). But we're using `GeneralReLU` with `leak=0.1`, which has different mathematical properties. The `a=leaky` parameter in `kaiming_normal_` adjusts the variance of the initial weights to account for the fact that our activation passes through 10% of negative values instead of zeroing them entirely. This small adjustment helps activations stay in a healthy range from the very first forward pass, leading to faster and more stable training.

The `partial` + `apply` pattern gives us a clean, reusable way to apply this custom initialization to **any** model architecture — we just call `.apply(iw)` and every relevant layer gets properly initialized, no matter how deep or complex the model is.

### Test-Time Dropout (TTD)

An interesting technique: keep dropout enabled during inference and average multiple predictions. This gives uncertainty estimates!

In [ ]:
class TTD_CB(Callback):
    """
    Test-Time Dropout callback.
    
    Keeps dropout layers in training mode during validation,
    allowing for uncertainty estimation through multiple forward passes.
    """
    def before_epoch(self, learn):
        # Set only dropout layers to training mode
        learn.model.apply(
            lambda m: m.train() if isinstance(m, (nn.Dropout, nn.Dropout2d)) else None
        )

---
## Summary

### Data Augmentation Techniques

| Technique | Description | When to Use |
|-----------|-------------|------------|
| **RandomCrop** | Randomly shift image | Almost always |
| **RandomHorizontalFlip** | 50% chance to flip | When horizontally symmetric |
| **Random Erase** | Erase random regions | Prevent overfitting to specific features |
| **Random Copy** | Copy patches within image | Alternative to Random Erase |
| **Dropout** | Zero random activations | Regularization |

### Key Concepts

1. **Augmentation reduces overfitting** by artificially expanding the dataset

2. **Apply augmentation only during training** (on_val=False)

3. **Global Average Pooling** reduces parameters and works with any input size

4. **Test Time Augmentation (TTA)** averages predictions from augmented versions:
   ```python
   pred = (pred_original + pred_flipped) / 2
   ```

5. **Dropout** with inverted scaling:
   ```python
   output = input * mask * (1 / (1-p))
   ```

### Augmentation Pipeline

```python
tfms = nn.Sequential(
    transforms.RandomCrop(28, padding=1),
    transforms.RandomHorizontalFlip(),
    RandErase()  # or RandCopy()
)
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)
```

### Best Practices

1. **Start with basic augmentations** (crop, flip)
2. **Add more aggressive augmentations** if still overfitting
3. **Use TTA at inference** for free accuracy boost
4. **Monitor train/val gap** to detect overfitting
5. **Augmentation strength** should match dataset size (small data → more augmentation)

---
## Export

In [ ]:
import nbdev; nbdev.nbdev_export()